# Fine-tune LLMs for CTF/Coding

Complete pipeline: data scraping -> synthesis -> training -> export

**Setup:**
1. Runtime -> Change runtime type -> **T4 GPU**
2. Run cells top-to-bottom
3. No external files needed — everything is built in

**Expected runtime:** ~2-3 hours for 3 epochs on T4


## Section 1: Install Dependencies


In [ ]:
# ============================================================
# 1.1 Detect environment + install packages
# ============================================================
import os, re, sys, json, time, shutil, tempfile
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path

IS_KAGGLE = "KAGGLE_USERNAME" in os.environ
IS_COLAB = "COLAB_" in "".join(os.environ.keys())
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")

if IS_COLAB or IS_KAGGLE:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xf = "xformers==" + {"2.10":"0.0.34","2.9":"0.0.33.post1","2.8":"0.0.32.post2"}.get(v, "0.0.34")
    !pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer requests gitpython pyyaml tqdm
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xf} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
    !pip install --no-deps -q flash-linear-attention causal-conv1d 2>&1 | tail -3
    torch._dynamo.config.recompile_limit = 64


In [ ]:
# ============================================================
# 1.2 Import libraries and verify environment
# ============================================================
import requests, yaml
from git import Repo
from datasets import load_dataset
from tqdm.notebook import tqdm
import torch

print(f"\n=== Environment ===")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:      {gpu_name} ({gpu_mem:.1f} GB)")
    if "T4" not in gpu_name:
        print(f"WARNING: Optimized for T4. You have {gpu_name}.")
    !nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu --format=csv 2>/dev/null || true
else:
    print("ERROR: No GPU detected! Runtime -> Change runtime type -> T4 GPU")


## Section 2: Decode source files


### File Manifest

- `src/build_dataset.py` (19672 bytes)
- `src/download_datasets.py` (12418 bytes)
- `src/eval.py` (24954 bytes)
- `src/gen_eval_bench.py` (57380 bytes)
- `src/process_data.py` (7030 bytes)
- `src/synthetic_rev_pwn.py` (49593 bytes)
- `src/train.py` (14396 bytes)
- `configs/gemma4-12b.yaml` (1158 bytes)
- `configs/gemma4.yaml` (1042 bytes)
- `configs/ornith10.yaml` (943 bytes)
- `configs/qwen35-4b.yaml` (1004 bytes)
- `configs/qwen35.yaml` (1006 bytes)
- `config.yaml` (5059 bytes)


In [ ]:
# ============================================================
# Decode source files to /content/
# ============================================================
import base64, gzip, json, os, sys

encoded_files = {
    "src/build_dataset.py": "H4sIAMlDR2oC/+08XXPbRpLv/BVTcNWJiElQkpPdHOvolCLLSfbsWCfZtVcn8RAIGJKwQADBAPoIzV+xD/eyv25/yXb3zACDL4lWnLp7OFdZBDEz3T3T3z0ztCxrcFyIPFmzV17uCZ6z74swCnjGxuzcz7yUC/ZDmP9YXLGMp4lgXhywIPHFiPlJfMOzXLA8YUdR6vkeWyTZ2ssHH4S35NMBY8UNy4qYicyfXCFYN5BInPSejcciKTKfs9sszHmRil37I/Zd+3pRNLBgjuE6TbKcfRRJrJ8ToZ8yrp+8bJl6mSi/rzyxisIr/TUP13ywyGCxUi/HBqYaTuFr2Ymv00UYlTCKIgwGFapfCy5yIaEsw1xDOIPFHci3ah5CN0WJV05O9sh/Dda6FZ8Hg2fsBx7zLCkEE96C5/fM91JkB+N3eeb5eZjELOLxEuh8e/Sf7rsP708/vHffnPzMZuxwH/4NBoOAL1jAgyJ1+Z23TiMuhvphyqJQ5DYbv6QH5C1jsLBnfJ3cAEuKNAp9LwdhubqnVQvjJSsEiBFISc7j3EEu4CDBeQw4YS5Dm14UcQhrAq8u5vRdEs3CmJXI6T3+u+b30FFxxVkH3wCBzhJA7YVxWuR7I1Z9F3lW0Lzh7d6ebTs89pOAD+Fpxe+CcAl8UCTgv3BB0OMkR9RIZYVW0+14QTCEXnatRU7A8dKUxwEQJFuDLIEXAZALy16uI6wgfZdjZE/ArDu/ZPsV1jQL43y4sPD5FbKFB9Oy50Y9bI21tyS8jOdFFiuyBiQbpgKDtgpSbFrp4/evKwWEL+5fz356f/Lh1D07OX13jlwhmM9YGvoJtNO3jVVkkTVl1irPUzGdTECQV8WV4yfrybH30fMnqvf4rwq0NWJW7K05DsI2P1+MfeyJDUj8MsnujUZrO3oU0014E3rxOPDCyalEd7h/eKBRdmHE9kcQqtkeZ/dpnvzo+dePkvHKy66PQbDeZctJNcxE79PbFbwFYrNrlMImFVWXXWZ+dF+I1Tmo2GrsK5yrfpwe9n4coebzbXycRBFf8kfJ+PHF3b+DWYq9CQzy5aDxbRfHq+bVi7sWB8rWXeaegnjn91f7YSi89WcgLsf9PvRhMI6TmBuI3dsOgavwhgH2fxypWv63RZSHY92V7aJxR4G3vvYC78qLJ9C/tgAg2WLsVR2adKwR3S7zllK25HHoT07u0igJ807l5qrN5EZj4rugW6h/+3eTA+5lsYli/6581TkVtZJnHKITwRk4vTDmPEOPtEh8cErBo+jXySrM16FYZd6fv/3TJCNQAOD7MK/NttVgEgSN+Grt3bkpz1y0vvAanO0O87/79k9X9zmfnJ2M3x6NzyACWHtpHfN47Y2zquFxzAff7Fer830YeyBfil0exQdDYI698xpJiRgfezlKXdv2AEFjNKwdAvDUVfkedDji95NVcnuoKHevaCImeqN1N+QHhHyOnvIVTH4N4YpcEHKYEwzmTLc5XIR34IE/nL0R9uDVu2P3/N2Hs+OTmrO8jfMkiYSaUX6fEmVyKi4GsEhG/0R/gLg1SUXEg4kGhQOuMi/2VzgmkBzGMJS+KoiG/aFBY2ooV7ZBB06sg47Mu3VkDwzgdPzWQ5ZU9DFo+mTtiZxnILJHr96eQHDWSZC2C+OMg+g2rZ8XLzO1vlMIH1Fl1x6EYxKm3T0NBenBBUXA9MckCr+PEX6Dit9e/CE0/NeL0wyC5Wzy2wuTit9edNEACxZcLf8QOiRo9dFgErzpoubs3SlbegFE1uLpeP+SxCCuXnzuRbfgqgCmBFkza0kqXzbk40kIBeQJYnKGcXLWQAJvuiXwXczVTJ+ON/BuwuBfvz78OAHP77YnCW87J6lDAFilibShkGyBzmch2J9eSRC/QxTE/Tq9l39NAulFTQ6eBH0JUA7lXxO6fPH7GLvka1wlfKgRrho6o/mxSLkfLkKfKghsCHZOZ0Zk4u0vZCWNRdU2UX6RPFUfTteSy6bxOgmKiMRirtJyP0JRQn80BDqmDJLbEeTqkInjI+XlV2Bfy7z8GPszjyoMOEqJDXxN7/NVEutsXBURxKrIw6i2ZBH3IIdMIUKAfB/HBmHG/RyDUkhYIZoWMF+Z6QIZLnoicIBYCBniiyq11a0OQRJDu8pwJVonW+cZ58Oyp10Rkmf3VXeskThyJbAMgishFwH/pvlqdlAl5ioJrugp0VMXfufDCHZCH+jqPcF4V+r92gP2B+j9CTHbANLtlG341mohe+1FgiuGqbKLi8meexUl/rUYKiGpWFYrpZzIEQxHMDmCUbVnDTljkNzGupSieQdLBbKFxZRs75dffhle3j63v7uMh85X39nwfY86gTb4KzAf0Ik7izAOwIUP1ciRhjjCxlfv3h+9eSMnZVBdL8xE4DNHkkQwQAp4tW7UMKMPByYZpvUCC1Y+sM1mLyHoqtdXDJS6lLKxEB1oB34wwG7lsKwU02EKPaUx21rRw4CiGIFzdkUSFchm9yopYAmy+zYvgOElK17DGHa74hln+YozPXqighcmuCyoidzLIO7XlS2NBBkGaUEZDxJ9e//97NmleD78bnqu+n1S+dMnlU99OorFLc8+vY685SfdaWrvjepAvgyUEsgN6Db2u+HBp7f3TI94AO1uIy6/uvzKJLODPBv71Ee9vBSPjlIj5pWViCDBEyR30gqLNArz4d5lvGeXkhuOqBvVFWOI8TNIC4Y00DBI2FFyj4qADYbWBRbkGbSGVGAoO0gMpEw//fDzu7OT46PzE/YJv7/98Ob9T29++vnErgMxRDc05Rg1RRLXtCcrUGBIabkLVs0HDUMCa9IM6qkyHvoKy2JZJOHwpW1sNDhmgJNmRwt7aX6GVxzWhzfG2Fr8MamKqLZbLzDvxCGtl9DlYYU1fINBsavh0+fFVI+aNztCF0TqfEzCeNgCYAD30f9R+Ra4J4qrIdlYMK1oWbGgPDLBjtgCJFPMmma0DeQy3rwYbRHCZXyJpWnVpzlAPdWs6MA0o2oYWFK18IYVbkK5mKo+c/acWY7jWDWbKTs1BK3kANrcpwuYBsMQTJ9geYscdwq6ZPF/T7xgocthL2eGRk6bfl9RWHcCdYnUkKbzWq+aONaH2s2O/SKkO7WG6MdeKSpHdolRB6CHBUn3akiSTvkpaJMxLP6hUK9LoEYUsboYGT8SKmnIUq48GaIFMuCFgC6FRA/k6j/+5QhipTArHbXejKlCm5IeHcKWL4w1WweuLAXNiJrhRm6wpegqyv5OtoySq6H1lXOxfju/CF7NLVuFKMb4C2No+Rp4grtPw9TBqTtRAh5vaLPZDAt6mBRhTYX2X1Mn9TKQVmyrKDUcYp7kXlRRC5zWWHT4SwtINmJfSg2GMi5ucMIr/HDwjykyONwNUVFnhgetwzX74MbksKPjSFI3M2iUtnQmt7o2Jfe3EOoVcZjPLMrBTFrQoQd3I714uI4l7kqCa9mDDDJJy4E6NdDBlZUmjmdZkonZXriMwcvt2U1nL4NXaSXYv2HRsO3KsT2MC15rqFNQ+m+coUEI5IlroCaNPJ8PrTFGucyyqzeufpOHecSHdl/wjNzpzz3sfsq0SJUUyRdtPDUlkJ0cLfSQ1No22IauNr+/SaygrYYJOewvkK8Vzovpi3l7zVtMrhFbcdxf7MrsjozFLTnfzlx2zWKAAFEsFuHdxcF03pfOaETbNknNnLWbitQTop/J9VBop6Cyss51kmoBggGsM3DoA9JUMjSAJo2YcvJapIa6982TVG/lCVclqCVbJauQq/sP0qXHtvEaBw0wEjqKvej+N8wbQ0Gb6+XajmizBet290mRMZCOLPH8FVhenuKpCfwcSfOeJTdhIHNPtZExKZ0x5dZWBxlpkWtWLKxjjXXKNnWTs72Mj0tvu9Fc2WIosTGWedtG0XpRcjrFNLjypo21qwlEt9DWIWnNqQ1s6wOaCOJfaSUkN7vNBIWJWEeYyUEXUi3nvaaj6kjqOb+Y1gPP+WfMZGFB2rBBhLDQGypbXMbwymrPKilyg5MW8sVqxIcEuj6SR+L3yGZNCKfgSjORg4MFxQ0X9ySHWBQl4dQSM8LX8owOptXYx7vxwsi7IncsT4JRagRWV2ACsMqSYrliaSJEiJ20BqD/R8gQmQM596X4k5qUcn8bQnRGGhQT3P8jOrCbfNeZWhvQNvOdrGxDWVjPnpXFF1KG9iwfnE/TxpngkR9oJI13aCS7DK+Opkt/10m7ZcgiuDrj26ivPyCmnpqAno6SROhp0NrT1dgKLsW4u6c8Pgj9ymC03bHTReu4+vmMHdRXezDYzZHXHJmM1iMvFRSrG4E5Gxsx+6B5Xuwff/+bGUhjwVqTtq3yH0qbNkYgvpU7IWy4UTinzsFiK+z6yTI9XuV5clscd7WrrXNjl+KBPE4eMIXEDTcnIjrZaey+U2Av9y30scn+DK69W5BxkSax4JQ6y+F0NJB2DXDRQF5mL/bt1gAn80IBIWeSubDCeSGMYLsKJ8veNSUefEbC0FzNFgxVZlbFe1nTwPR/+N0MVN/eK+v3dnss2gNdpqbzjBLUtCurUY1Pj6woJXGpFCpplVXRbE9WjZ3nSKvG0h5Jgl1CcJbgJ9LhgY3kmZDRMDKrdkLjAYP2uFlqmKSFdaLis1Vyixs+BaDbkPrgaoIKETGQlA56jZWaYytU6BhSmq2F9b2H2g2MUthqajAlK94HuGHSt0/e3VJnW1rbWw/ovTx17eeLIPmY6APK2mMYxR0sGQl9jDgk5flmf79tDd4kXsC8tfdbEo+FH/LY53iyaPwKwLPz1+9Bvb2PtAUZ8rLkAxGaF8nCj9PY1MTD3jhzl059k1L0Hd2WxVc1kyxJcl0HspwJTgx3eyeqXS2MWORuEGKhozZuwiwgdYyDrMHAyGpU/45dUHO8Srqd9TX0VVmymL3PCkofYKSbXNNXu4ObuOmL1rJnDbFcZ/duptbkqPMITTfUhjoAx4fmhOxRI/+kfdrqZd88OJgdFJkc6z49Nh9tHPI2KmtAepHLIhw1W8ZiUxiZgEUYViNtlJLFtFV/0PtFi+mgpzRQHet+OavJeWdUcQWR8HUbVM7XMCeSUgfP+guqM7eDC9Cva+iI/cmRWfjCGhkvyMuhNRthSb4Dglag+zqc8nUNGgTbHOMLs7UTLO591ADiC9m5a+FQG2gusMT0XCLoq+N0+J8nR56I+ZGQ07IeCzWNhXzOQGbRRuPeJFhOnPvWwnnSskinZdmPR6TV8pUvR3SAt3dwGaRa2jQ9GKR+lsxKWaVXZAgNs27XjVSHWaqUzBgGk7m1mqr20GUPaneoxk/K6gTFOsUrKVhFhCVX0qXtxrl3g/cialPcstKBlOYDnN3GIGv7YGwrfVy5g/Gwk9NHSk0vJ6fjZkXshhjB420gB/98PaR7KBfTb+faAdLNK339Rzor43qGPI3a8HGgG36RETcWBUwACcdmyAPdtsF8vMgvFxMWdzabsXMsChxMDqfyThi6FqRHH/sub4xB35IbxvaDvG4ii22tmyW2Poh8ugIY7GCqrD4Qru6pgEyAnGFCG9X4rL3cxsCybY+pvJ0+OyWKiKpUm+1AbQvrc1Xwn3Z8alt71Y7QwtroS12ooPiMgm9vJ5RiXexh2LZnxmJKmLB1ZJ7dkt0hxtqbj4ydG+N5UClQm7fO+xWWsE+TJDq5434BBmiIgneb0PmE2dekXml5EIw0SA7FeWMD7iGuIY8oJy5x2zLTJX2kB1jJFs+2NbWVgGXtrUUolUul/w6G6qXdTH9wcUTh+1yIUX3BaYAjWdbc6jC5eWGu/xxGDtvwGlVryuiwGgYpMplpNUJZ6n/8/X/qIXUVlmzk2C1rcL0hyqAuen+SVNjYlKwJKBsen34Y08bwtNoLx+uHKN3PYcCyvEsGkurq+joJkpctRZe0joz6XFW6kI8gKTBvHFnxEQkUme/UblBq49K/gTtq3BMcNF2ysR/w0PZvJ7l2F7See4mtM3i64WKq5zw3QA/M3U8Xl6IeTfaKvrEr3iWwDwil6X7V2LYeKN7gowwBajcU6t6lUfg15qLjIHN9iZYqolCLoamrJMMI69oFyEoHzq9Def2wpgJMmhO2oHTS1hphmCbcVgwhHBCOnxZgGgqAZ2MA+PWIfVv345exUh+6sEMVKunTq4na2y510lZwY+Dd2pUf6DOqp1mCXOm1qsbz7va1pq4jJvWVPi4O5yRrJH0U+FQT+/IWVm8zqx4X8rO+d9G5g6oVacTcRyxyM96AFDeXN2GbOrpDbaItcmCRm6XMWnHijw5PKT8DHphTfDRIxUH1MPUpldzLGOu4r0C1plIJTBpso5ArKPZtRraPlHFNYLVwF4/H7xDqQjcd6R6WkW53GFuraOkQ9svFqIe1GLV+h+uRIJWuAsgY1bjKVR0bhXaXTprAQ/3sqNm9XVu72KiRIAQHEChW2LZz+cMKFMPCi9KK1uszpt/AXvJawpyOA5k3E5gksqtdXaWYdtZFcR+qq3BPsPCuw3wk4dINhbnpHvopoutmDSPk3boADl14Cbk61VLVlzDZ7b1T0SgkyHtwEqT0l+pmHN3tC+NGd7kOigoI5hVJ20nGF2KyglYx2UgIwCkNcw9FXv52wNa41PZZi1lh7l3Mpq9ta0RNGmoBlg51DI2shuxmkk07+7dW9lxiK3eP/t/y/jGWF4wKHr9RP/0xlLuetBknf3pjxJqGuDK4b3Eso/vPAJH95fzdz2/kRp42tWR9yrN+n8mFkhTkhUlYcy+J5KHqYHfUuttlWKM/Ig/jRXcV0CzI9nYyZaK7iFrGPLQi5S5tjfm0noEqMGx3rRxRb81NMEJ62vSDMpnMvOjZOcqWZCNOqcU83DRTztOXP8kDacjET8ipacnQRS8air9HgnEjQRta+ldvwAz6qySEoHZ2YZkX8fU1YSwX4ifIocwIfi3CjAeGovbAl7Mfg3ZbmAIuPJC5maV3SB6mDYzUWF67pJQGnccsjPMKDpbLdgEAs+gcf6jHq7xOgaEPiq7NIOLBMIMEmTIqxW6YsN1v1RqWY7M32/vqm/1toz7JOn5hSfILTAwhk9+a4xrg9J6SMYI8cMnnSjN6ipcAsjG3rb7ZJNSGiUxYnHbaSc6/ibrh9TuCyC6U2KEbHbQ8gI2Et4mtuQvZhVD1edIUUVP+0IX9sgv3xRfmdzjSXnV4TyYYR2BOZ7rLfg0YDIA5LmWCrkuccV00tq6r2CMt7+CfNpI4RNhMAAA=",
    "src/download_datasets.py": "H4sIAMlDR2oC/+1azXLjuBG+6ylQvCyVFSV7dueiKqVqan5qd5OZ3WScykF2cWASkhiTIA2C/onLT5Frni5Pkm6AoEASlGWPdleH1cGWiMbXjUb3BzQIz/NG76ikJZMkzm95mtOYUB6TQrCCCiqTnJMyEkkhR+/q9lIJRDm/YUKWJNbdSyJz8iYtaETJKhcZlfiP/IOXaS43RAqa8ISvRx5oTLIiF5L8q8y5+U7FGvSVzPyWScZGK5FnpKBykyaXpG74BX7qhkZx3YK2hfXD0UiK+/mIwEfJyus4M3L4XbX88OZzePa3dx/JgpyJio3YXcQKSX5UYu+FyMW8K/eBpmDjaBSzlda3WYW3idyEK5qmlzS68jnN2Fj3g6GeiXs9dFIWaSInRG4YJ5KV5isA0SqVU3SLshZ8pkQJ9Fl6qq83IR528S40Ln6a4ZmPYLISvOUEZctEwy3U33HTpR7re/UPJrkNBpMrEw4u2YU8rv3A7sDKSIbX1E8ky7aDf68byHUFxoOOGeXlLRN6Rm6oSPKqJD98MBMJcbZhGS0bX2TlugSfI+h0DUq9jJUlXbPS0+NIVloEwzFl3McfY/LnBXm1Hc01AODz5cmFxsChMS7Rp97WHdSIBac75UDltdJHnd6/nhA6Mmq3dhsHeGMC07t9Xoj8MmVZ93HMdMY1PWp/0Baodma3b5mnlUvVKqXrFphrJN1RSJjalk58EEabil9ZfgEkfN6DgfkvUgx9uUlK8vbsA7kVAFUVNcHkN0nMMAtIY/REIdlR9ynnEML4t442w1JhJCH12GXJBPCQn1eyqGSIZLHwMKBmrfYpck3qbWPTkBn5iUbXVzA/bfnw5mR6Snz4rQyPNpDejEPojU1wFiLh4JFzTsjydPb6ghhEIDnSx5pOp7W7SkmFDJHfwLX4b4p/fN0YY8C3Ms3bYaBnkrvmCchInb0Vx3k70ZAAFILfBTyJSzNjhtO2s2bLIUX6cTkhGIoLT4Ha2kFxxVGvyG+NVqRBkheQh9ZcQJSAAKElWW01rerIRIprtLbzyWS6O/0nZHkxbsnDiBQD1BJdFmjEeClFFalVbdEoeYIbzEcPy+73BFk0452qwPcxBqdxlRWl/9ATUnFp2efNbWsnQ/JgEUh63oCAthkk9Jee0OOYfEsgiB1W6yj6dkFO9fyylBYli9tBSwIrnq28WEHM/O+//2lHzZw8KNBHWDJoVqQwvf5DDTufnq4eIbvGdvJbgdTNfgy0KI9ZKBgFt0LSOSigL1TzwIRk9C4stRGL0xP4OKiBA0EldPYzoLwFlL8bEOJHeVYwmcjkhiGRrQXNMmhwksOr2fdtchiCfTlFDCH2+QEBDs0RMpc0XcD4fUzBuBy3vDu2ScRMSCAsE20maew7KJvAWPR4gRUs0/r8cAl2XbWe1kt0i4d01vVz3SxjLWFrbevKN05odahTti/eHZIxDVdTo2V+MPJZeZ/z9IbpxXsg3o0J83P+UH999L6aqlbeQ+OZx3N+zr98+VLcy00OXx/MQB/VY++FhNa480Wk1o/iQzLbinGRuDY0uiGM7i8ZFEpRBdN6/yw+e5MmEeV/SQT998nsrQ0T1DVg8EHpCG5ewebCwWXfdbmshUJMJalRCKK8nNSeae2xUZ2erWOnt19pn1XBgh9CPQWwHDb0vp8tmz3ShRqSGk+jHSu5JfgoZdC8WBAPAbyxeztFyzKBYOLyqzQ0KENqoEczDFUn2WrnTg7bm2sdfGt0TXZ12E2hbRpt2euU38GVB+VLnQqH5EjcVpa4PLl4UtDb2VbgWQyJx0CfkjvsHujugTmc8LEO1M909pqVyF0Sfnr/z15J2IF8OTUOm3l0NLg18eipcPC0aZgGHQdOA6Xm00dQ/TJzj/MoyyBbIfKVxjk8UT1Zlb6Aqwaq09+SpLaBekiiKqssYyKLkLEGqMoWeRZZfVYdP76dITP5elIqAePG3xC44Bdz6q7SZT+eslFfzlA2ytFxkvF4AB4/dlYCf161j2DhgYME1gIUxCEEAdhty9sNjn54eIsqkDFs0cMVkgi/nH8HMXzx1aUh4Pq2lePl/DUC/y5loB1Gh+QMKRiP7/O0qfQGiKMn98ytju5NWvVkaxWRVcMduLXYhz0MatAq2L5mp6PxZm7g4MetucGZMtcUh0fHOma6AuXwY+cdLEdaPKIqsj5/NFVG+91UU1w5GUeBt2qqw9ENYn81zzR2/S7E0o6UQ9dNkbgvZB5STtN7GOWOAqoj6aSX1wN7E3oF3ZAZ1cZKIwUZ5RVNAwMYXDIebTIqrvauoGqgxqSXb1AaA2d7GXiMdZU2+o+66o+66sjrqk7WHoDRYOO1ZiabS195JVwlADUhfUJT0rixgtTt3QT4iI0kq1KZgCnkp88/f/orUVAwJTInOWeGnlR+Nqn+zARrTMQ0swzuJRrecLJGNJ6yO1iMSn/cj7mtCZY8Kk/4aiBCwZQ04bURA0J2KKPweFBKe6QdJmb+lWNj8qBkrLkGnz5YHnt8cq5h1szg1R0x5ExzX2z6RqyrDBL4F9XiW3dnFtvlaHujjWF1PIvy2NrZmvtEGnxK4zikNarvBdbRWrTJkwgWvWXjDw/D274W4TlfcXrN2wCvdSDmdUpRr7dJ9Bz5g49piiuxpyJbB+fFBFx4XSWCxQs8INw5JuDwoOZwvHRzX7AFTNrEXEerq4ZdCHqegliNynRr9g+1R6EDcn8Nof4hSOlb69TOlVvlAvaY1nEB+sbT7Ar++SotwvzKGm0r/s75wzeLb/70+sRE2JaYzIsqEyAQUoGJBmAnpbD+2e/8kd4Rs/7VstaS2JXv2GB2BLYK9T6iHUjWRsF93wmAO255dN94qpnbpdIRqg69jpsWLuU77lp0XbTLpjpRHHbU70Vdune+GX2OdsyqA7v+N/Hlr+ioniPq9y5DXui8ddkDtHVG6oJ1npDuAdw/SHGhDx+j7OmQbkE15Jmhcuo5Aar5fhuiTcG0gFZccO1lDXosrCjUe5oFWbZW832jefJkr+GofbLrrrjcy9p21D3ZxRVRT3YaDJS9LByY/abv9gyzs7etd7UuXNe+9sXb9R3L5Zna5WEPWPBaG/XhxW40gvgNQ7woHoYqdsMQd3JhWIev3taN/g/WxLJKgjAAAA==",
    "src/eval.py": "H4sIAMlDR2oC/+08y5LbSHJ3fkUtJrQEWiT6MdKshzHUrEaPHdnSjCxpdsOmaCwIFklMkwAEgP0Qhw4f12c7whEOH3zxwVd/gj9lvsSZWW8A7Ja0szf3oVmoR1ZWviorK4HPfnW8rcrjWZod8+yCFdf1Ks8+73me13v05il7kc/5mj25iNfbuM5L9vM//SsrtxmryzjN+JxtsL1i8RIeq5rhkGQVr9c8W3Kozuas5EVe1vBTbdd1FfZ6P1Txko96jG0vCFRVJsccJgiLazYcEkC25JtNfA8e43lc1Lxk+bYutnV1LBqGSb04XudlfAhKkm+KuOQSzujAaPbukmef39fN4vFW4B+HImIT13yZl9esuMzgcZ4uFmkC1LhmPK6uf7FpRAubx3VMYI4lzY+Jm+mG+BCXS6BMxdXzj1WeqfImrleqnFeqVOq+dbrhvUWZb1iSA4uTOs2zisnGOV/EMNs8TWrRB/DgOEJ3kM+itYC51ulMNb7EqXt1eY2CwVQtSFyy6vGrhBc1e0Z1T8oyL0UnamVj9l2e8V6vBwiwdR7PoxnPktUmLs8rn4oRzjViVV3KzgEbPmDrtKoniO1UTrlgpjdLK+opmvDPahsTtn4ULdI1j6IgRFHLavnDjpmHLPCwgGygArBJ4BUiwdcewb1MAVpe8MzCM2BxxRZm3pLX2zJjExwW4vIqf53iEhagjVhiacYWiD0+hLDItPCDqaTHZbqGcVGS+km+zeoRdK4HLJO/70dsASBrWNFp+OUXRJZ6W6z5hKoHolXSB2ToDwSNVUle4rQgjbA8QiRmYEDyTQqPRZkjn0A0mP/l/Tvs0TM2u1bSEYQoipLcGRuP2Ulrqf5JeDIAhE4CaikAO0IeyJhRzZzDVIgzu8veHx2d6YaEI07Q4hemyT9jRywLAijSQOpZFSWP59DzPTSi2IfVu7L2YdwR80/ZkBWBBeGehpC5YBTGm/hKYC0xGMoJggHbpJl/ajXdVU1BT/IoIkWPsnjDfVE859ckrsQQ+NUMeBEXwuIy6ALyz77dLpdptnwaJ1w2IBhN5EOahjIWvbQ17TrerKkWC4KYebZIl0rio5e+J2oqD2mw8HYa132Igwxb89oeHPIr0LTKDwyjD4EWcIKGbli9m8qRLJYIAIeFVbzgESqIvwhs5kCfcMlr3yN8vQFWBKIGaQUVeiGaJWRGqLrBkQGTZrhtUzSTnsNgyQxUjTRbcLALwCC5JFQN0JPn+auHCprLsG1WrXO0QYI1T+Oqfh5nyy1sm7QdD6jq92kFgGQFLCeCfbeOar4p1mBoewTOSBaSqUvOgp5im64CGxKXdYXY+h7tMp7Fu0MYWuggk57+/oUeshE4gmFIeFXlSDFoDhFSBLogXQlf93cxH5viQDAmzaJ7s7Qevym3fKBHBbpkT9QijK9bB0ysbnjPM2Pr/Jxn6XuyIrpnqGvdNcmF4BzA6UhzWpBXAAXviN9OPYe/RL/nLfrZqEHzL0Q/Nq+vCz5GEe6ipT1pm5a6FWiJLVp7HRI9v4FESvpstSK3kbZYuzboMCTCvvFFrY0blImGLSx0S4tukrzOXJYswT7nLzzGUK3B4bXVVsy/s0fuPbkmaX2avBP2ZckzXgIBI/DQCvChuN/sN2DVdQV0BkTBla2l8dlWOI1V4/Dc4n58FWX8MiJwFe32QIL7p2ew90dVDNzjuvbUeEMA0Wz2v5M4Mo1jFYgF1yu52YTsFa2youEsX+i+lbZoG9CgGM8CYzbR2O68Ml9zb8Q8sUoP5SeH/TGrodJZ+X7QMQrp4I6xKCNHTIVoZeASg8Be4VI1dcO4KNbXDWE2IiNRNuwYP41BjVFG5pHkHRg7OR/pUc+oTZzU23gd2ZpjJja1IPSruIrrunTUSJdhm0XbwVzbQ+tBYjZn8c1KB1L44CED61WNvaL2AphaSFk45xdpwgOLPnBcA5AC9sQTdem88qZhtYoLPhmeTpVUS/4iOwWFcY+L0AktwYZxX8uXpaa08ZG3HmZ5tCxhj7Za8Q+IGp1fwtkEAe88V36BvW4FkGmey3mg0czJHoA47x3I6JDYzaOWypi5Jx4KA3J3W3JvCqichF84/eXhC1oEKZUi+0dHgnoDdnRkAAaWb6sUA0SPZ3O/yT/gSgIgfTnD5GQ60bwZTcEcnKdFVBU8SdWoigQvUC6/4/To6aRDgzTn0WIdL33VJE0KvwKYNZ8bh1OcAGZ5DvbIsQiPVjw5R4IqEOjI1XjwJ5OgIDGcxng08ID0Knm4SLN5vF77ZR8r3+4m/7Cf3n277w80wMD2IWlk63QgNdH7LqcOuzAM9+jiggGGcxCcE+baEdUI4QmpE5jYARfgV9fJCmmwU2P2Xq896cJ7ihOwHUHbsxmctvUk9lCb6Jvk3S9K8w3h2iD5i0d/C7tmdWm5kgWd64DsGAuBgyqQ3f86DfyvR6Lj2+oIymn10yj4Gso/vT36WjYc4fNIVPoB9ps8HD6eBm9nfcEgPHwB+LDiMei03+Lexnb1yxJQRIUJl2W+LfzTINyCCpR+gMc/tQJV1+SQHI+0f3JVrNMkrQUFgFdtiHvmt/mhGwNBmDX4W02BRMq8nZl1HhBJHHqTRMJwtuY1uQZGFA0NcDwa01sp0F79cxg6VLAB6VmcnAMNmhBvpYAtmmRwGrJZcumh2Ycb2Al5Bc5fXKHfQJu9FUn5FINRXUMhqdMElnLNLuJ1OmeIjtgrNBJ4zr3My7nxJ7BTNFvnyXnLqvzxj38EiRYRzJ+Sn2ZxtfqpuAbpzvzw6OsAmi2+Yil8/P2bh8+fB+7JVcM3nMbQiphODA4rEMXa77/N+oEl6jBS9ZyIw98abY+oQyc3u/bPL6mKWkV50pfOK2DXJwcLC8glqoB++JuKR+GPUomN+9Ng2rM2OoOBu8u5NJsg2uGPeYpHazUgmPZupUJb3olj1E3Ju+AROPAphvsodKMhwZ4m4nYqyidwE9ZJj1H7GSzxKwyfPcDF8iueSFLLaOBrFKArigbieYkfQnPhWT1BYQC6z4PJ6K9OpnuJ7mcgyq6A82WcXBspHNJ+q9XOJpQ1zOaEEeBWILG1/UgEpRL4WW5GS4uFfxnnePxoiDwYrL+Ph++jqSqcDL+c7u4NxKaqoNiYoQzCclDwDJtQyGSlmGcy+nwa3ITzK71A4YvIbWnuHZKW1gDa4i2h+Yw93WaJDI4gWRmRVdw4cBQKMmooCcJIYMAHtuB0XYNBaQsWxYNUM3qVzmo88lbBeaRfwA/cLA85n0E5pSMFhUg9CoF6MlDqnrc8QMqTZtND8wcPwgp6aCI9YSnRWQXU4Ql/GhDIdEIT/ULPitMBiOMoUnV4ol94RsrDoxv1IChEY2gympnho7DcXjxDDxr+Q3mT4irhfwMC+NfCy0YcthvEYbvBMtglYCo8UgFqSn7By4rqVLEBi2fbDTnF0EWXYeT7tIAa+I94xAXNh+UFsIgjIUXBQDOOPJgP8HrxfgpPB5FmaxQhajajzRgUFGNVBgaGUQcKz8HelqDga1R9o9MNFQD1QQ5tCw8H1En7LEGT1slEdpt2TmuZDzRw1D+u4ABbuwPa4Jum7Q2qyS69e7oH4wRWFD1YgNYX0PrTvXfI4324XoPzwDN7sXtSuwpcu6pSmiyt7RP6wZj+zbb21TajKx/ebW0t36NAe9F0PmZxluEiLA8D7di7bVo2qm92PH6HU7CYFcJTTMgPQW6LCZQJEteTArqs+8v7GYc3R72Fy2np+K/gCCMp8R+rAixJHsP1OsamKFp7tuXvsPtYJcAFhxn7A0iEQz1gbsavar9uAsGp3GmCvXM3gEQ8hIdC3cJkk1ZVmmGcf1K3O4rZEKwNcHpwHS8kNCNUOzmBe+KT29xLFCChEJVYtxCmCoXZFmdwtdd5Wkd1GScdPrWYLKpqXlQfLccSNiPYeJlGKOAqkByaEgTcuV/L6+bU3eYAXTlq193lniw3dcAVtgP/lObDjjYHBGTh9KgTIVY1VOBZYI6HCuoY91vfBRSMbjVZjQF7ibt14jpwbJcT749vAONYKV/nMsjdmzmc/TAjpA8/YChQKWHniKtzhmF3E0D1zVEPHDD0Ms1VKXaPsDtaDIXQxNPV3lSaannuc3qpWs8YAgveGB2ceAnii8KFBt5r0b8jdmRCGOqSowV2s4X9GCgTJas8TfghsHZ05AOgknk0EdhDUJ2D7cCQQ1z7acfYC1ptZjv0goNYXGzXiMEshUPgdZTOOWx6CzjK3oTSRy6Udq4PWKm7jbaWI0yyiqMGHaSQOqB6HETIMXCHsOm2godnFRrnhi7NlpOdZ/llZvBA50aVG94EHgiiajvD9qppfhtK3E5zuFlt3WtbOQnsCWWN6Q4JrCOtLUWm7IguNVbo2fop6KFavAHs1040QrW0AmowwQyv5Fwr1QxbWWSlpSorE+DwnkyiqcHdvOTpclVLO18JrESdJ1IxhOlHu69QCuwdwgA4CU9kkkYNHmnjnqCqbQhmUSJWWdVi4qRM4RyAQg9bk3WhuEqxF1pa32w1RVy3dxokH8IkD4oESgOReKkw/E4kAaCkTEQRXHAPZoIq+A9lSYaRRs+hy945XMMI12lvkOfuuBuKkA9Kqxk3xxy7LELFtJ9piSe2AkkBpISh+XZTVL5cM6Y3vP7h9ctnj559/8Pr6MXDV3/z5NVrfTHnXSLdtwVd8iHp6XkIxxgo52W6TDMK1s2useLhFtxfcPE9cV7zvn3zDcP6b0Hm2ZsVZ9/kV/hMAS0JGZ+LNMkxS1BXybnj9Xm9KvPtcuXRuXO9peOGwiVOYFVz9Hnq3BqLeU4yerYCN3ydzqRZIBcponAjnHbJhPomL2xkZX8FjXAlymkOB9t1XLAZry85+Hx6JB0Z6OIYUQGUiq1xuZzwA3X66HwxOIEuYa/GIgGws8XsqJMG3nEpbkTBEvoFi1Y+uNVKSCS18H7v7P4Xvj8XUiluNIXesbtM1mJqZ7kV7gE1BSEQBPdYKK341Txdwrbpwznv9ItpI2MOJ6KI6A4QmAk7MqPTgubGvkE0Z4id9xYEJqhKqUGGDqTyaxU7NBAVJ8cuOr92puq47f8Ge7Of//Tf4H7GOJ2AMxIeqHwKpCdpgwY/0iTAeo51sEcChzFprDV6gPfxD9CK3nc5KlADzH7+9/9k7MH9O3jdTkMxxJhmTamkuBmZAozNgdZixt4CL7rnXvAhR3tDiz/EJQIeMUed1KlaxR04pT6Q8pXbLOIiZRj17tYUqgHrztPsTG9QybTu/YTJqXXrD6Q7oOZrvX+1BTuzhdOFwVmGBtRdKmUu64s1SZi3GaaEIMVpgXiS1BlxksT1CV7+pxse4j95t9ORztOVb9aVkWJ4IhKF1iInBbi/s2ZhQ5h4FJ4u9pXKRjHKpmbrTJKVEYobbaf2TzQjXI1Xs0xmXbpOabYTT43FS/axhjRVoC1ufgJwM1qAN8/yDGTICKwnnTFqLNZoKzHzz8c7LUf7IAzDt5mnfFaSDdfLSdHldEOKFmhri+iSjmaWxYekCiWrieckzaATg5XmSeM/NgkaJlQKFqmig2NTinq6z2dMOsagJxLAgCIS5+xylaogyVz3j7PryFx8uj7YIi3xlHWoVXrOFlXNIOUkKTdTTdVVj7z4UTiHLjc0fRu7pnLdtWfdca5AwgaN2zW1DoHEA8ymPnG66CUpt3Mxa4WDf2xkQ3evm34P9DGIyFJzClndnsNlls5nanYxHuqVr58HAqfAiArYj1jEITDkuIHd+5Iuu1MrL2VyMtX1yKcNtradUwuqMsPKcXcvHVK8kECRh8K0cSGhjY3oYWxPo59lN0RP25A0L2508EV0tYIxzdkFXel+w+JS8yoJaaj7CMK6PVC/oriOzvEyx/CrC47drQuUEkecT0mmjD8b5aODhec1hioG0vWP4WWjFxgJba8qu2srHUtMM2kC0FErQV0TxGqyIaXDmzRgTVpsKzhtpPm2ijZ8AyeY9zKUoiTUumqyBI38/DHzfv6Pf/GIJI5yEb6+94+eDKm7DTDo37yu7NEJ3dIct/aZKUYly/M928E6++m8P90zn8pG+KAuAP9KrNHd16WLYpRBZrmPTDq3WaQnHQqUDMu1sDpIHRMcw5LVptkGrbos2vfS76PlRnKoj96UfJCBF+1xvcSOqPcbTNya246X2lEp6iJfOBGvkWgHzGy61hQTjfzUSn3H1yQa3QSBpurS+XeYQkRBdWkW7NezZtvknAszKoo4qfWik7+ON7N5DK6fpeYnlMUJR3RVBhlFok0wsiD0U1yxwl4lwwcUx5f2EddgLLRYwMLblZO+whDkYYjPjoSYM6LEdAJDpxOJyRRDD6dOhsJEozx1NwR3vO7kQnA7mVVRt1KGtcimybeIfHvCIDiELdFqqox8KWpggB2ssjZcfSthU6+5vMCMZerKgTpaDWaDBZAfgH9rUunb8ziL1rHCTePf7q7iOjqgoyI5Ui5fgrn/7TmcZeJanE+N2XQEWr7JopUTD5Ay/0/uFzfTSZ779eYyYA1GNc48u/64f/SbE3XCUQ0Y0nlitPiV1EW304GxL8TxiXUcoFSXxzFssLzGCDDSas/ebXklXj9seOeeiZ92p/0qkLjk357CtDtHqsBGyyl8t+GYnkbhyR09SQuchQigqolqw9SVHfAErECJwDekGmxW8vh8jpFw2yY+eka9YCOb0xESqSsG9EdfnX4BG0o/TpL+6MF9LJIowwPVZ1C4h4VHz768H61zePyNflyl6lHJMjx/ubePvmJOG2Vv6LEj0izZFqhEMLJYKQmtSCHxgY2kS+YE2NINAVqbNzEcvIC4otDjhs/TLUUFV3GpL7YQCgAV98Bq2pFzGKCTsg3WNXza1sJ4sLBzy6aSpcIgkmWtmt71zBjbbk8eT9Rp1vCtgUkI1za0xxYot+/FUtupmWt1D45Z5wO2SmGAeRXUnmxgjQtakyk7NtOWGe0W+TCmpiNMBTTEcI+QQ1ghyBvKOZTVCkAuw7MF1MwmfZq+PxVCuc5BSkXnVaqLhAzIpeP8fMa+v4AD3Vq87NOxUEd/B8LMync0LEra9v+QUb5R1F0TlwuclBZ22ZDD1FDtBwmhlLJNi5e8tF0XiQYr88uq9wEKMMf9BNWsPLSftsIq02ZYmmA0U08bMj9HWrS3JDF958YtRn3y/kyQldiKh94NykH4DazONhI37uw3TWTEQ4os62N4XcoIzXlshmkBmTfEw+lyUEbmDQHp2FpQWNQWajYYW8y7e4y82xzW5unJZot1Ruo5MRf2cT6jOVapw1Mpzk5AzzPcZBvO8eirL+AERaDHO0FRQU8xaV8dejHTuB9MRvcxWy2QpHqsAQ1n8TrOEjivgEnblnFyLWwJHHbN6WDibm6Dpua0PUehRtssBXJbgPCGijZLC76kgXiKAAu6JdnblTLkr6vltigi5DirO5EVX313o/a3IrYiRN4dbZ023HtCdeJrPIJp2wLA/E31V3vNu+ZxQa6yC+JH2QaA3JhDs1hufAb9EL1bXvmBHmLa6PbJ5oq1bSiIjgVzlnEIsmi1YUvmtjYlurFSmOMWQW5FnJR5VcnYtoXsXrkx4KMq7MgP9YIbYYoFdECViHbBjYy66eshESYQn2hJwezakYIDV7MiYFClcz6cXQ/xl5nxwi9+kXzHwRr0K5E8jtJZ2DYMpaTjDueG84y4Y3mk5zl8mtEOe7qe24jV8WzNP9IT5iSZNkmMvPLSCnMEet5vyRNpnwceqYAB2MUT6c12T2HMgPxYAM6kIiU6/0LAvovA2Q47jr46u/+JRwOxB6m1f4gHD56Mds+tBX3IoiQHIuu6pjxA1UPmbnoo5u9aMXuWg95MMxxhDero9Omuz0G4n+L+3uAzdXnYNtekxLRP2NXY8oHRRgzIp0FbNESPhg7Huz7I75nFbSFYALjzHHCjm65lSPlhH6MaB5neKUBtttssdzW59wsx/ADUT2X3R7D6F2azxeIO77Vh6nH5Z/JzWcb49qysC5uf9NLlmWVWTtFZKsXN0cj1Vw1hLQD4arahskk1Kc8+FtBpN6B03twkTkM4TpOD8GtGFWeqwrq3mOX1KiqtPD4hoSlOiyBRLk8n6VQ4bmdQCtyxl2VOefbdYymbXI+nJxdGHOXZ+voD5u4YO7t5rDt3Y2wmVQr6BvZ9tZGSZJUOq3fbGLPkyVeQp9IUzsmSR2meOW/JCXzuKsQeNOM5ABLZ7cezypedh7Iz3pyfBvLTUQ1Arm38jBVD8v3EdzUIJuF3yuYLGFsUZX6VbvAVCjfIEolRY/HpKl4uEt98xIrAHLMzkBaYwV57tS0vUhg6oAms6/+Kd67upHGNbebFC26L8cppaGpJ04OYdfY8nTZuZSwzbpiI7zqNOu/WdgKBPdAOP3ayEzTfM/YTtM3ctpnVRgEQqDJ6c9dShH3nXECZ//0f1HMkkbRlBfv5n//EdpI6o/DeYu8mfmXsK3bWmc+FIL/Laz5i2XiX7amfeNkaM/TAAF+yAm/JPTtZA5NbJSe+6sgUg5Vm5AJKvhhxfqDEmey8oMwBpB7zhD7oBPQR8PaYZlaly4wy3bMaE0HF2+h+IdEIHCybMtUFO8ttkOSiy5dEfSTqf7XAOhtBRygDITRFqNXBliLb0rRe9hMm51djYXE60iaiVrSDRnSEOYwGtIac3TKEvnFI71qVH7IhSf8DNyHc6dKgBdDSnFRFTPDG2dwvioCJ0itSKRFo0fpEqkR1XdyhY14VX/Cuy2DniDeQn1SJ5mkp3qnRZ77XML7rZhj0gkLPdbwpwKj99evvv9PnOgnMTrs18INml3BzDtW+yMOt5PewKKE2ys/F11XEDSFd+8rPV4ZZfunTV1cWlCjl3fm7O5s78+jOt3de3HntmU9adToe1kVU2baYVrITjBamckEfkYwwpkVGsj/dR7u62lNqsNf1jbHWWPk1VHtYT/WUpLIJd6xBiHuYuI6d96c9TX58qdklC3he4lrfDwaNpAS6A7fOlQf87WlnPoLKENn/v6f+Z3rqJjvrVtqot1O602jsyDiIiDX4QI6WSk2xZKCZN6QTVLCPemr1ktFXTOSRjD5An+Zb7nh5ojKtNKkbnTQFv7yPcicoezipqzsNS+V/0O+npyVRf33VLcmiydxOXzIfzVSqja9xeM3PZupXQ3xkHIjqAPiPb7GNz9wQl7zGJ1OO7/2ynYK795yXtlS1NP2bOM3UKwn0tWFyR+SXh8OH5XK7gdleUos/51VSppSCPvY6vjmtwm7UO8RPsMUSAJzyh/qrnvSaYTWeiI863vPU5x3vDU/PZvgkvu5sSsN7M/FmSwZUOz1pipn9t+LrYizyFJAKclPiN2OmxHcgh9OHV2G08wVB2H9AejDQdCMsShlrQTKvpeAm+Jz5MjVpZH0DuvHh4eDmeXTYyyZocUmvfZT8gr65V14XdU7vB/HZ7UR7Sh91sBOsbsbAuk6wcTh09//h01uAbyaB2Cxhigw/4Tb27uInYnkdX8QlyMD3j588Hz18/PDlmyevvNtmfyQ/Q65ehlWfS/d5uAwHf97XyW/hpBilReaxkjOUG1Rn7U6h6FiSk4nmW6CbbCP6eih9oESCGJ/eRpXXMi2nAMaIDHXahAqR8eQDovkGJprjayWfB8qjkh8HlAjRD6JU+dZHROExlPwz1s4Jpbsp5fhKYsHp8x/dY51MQv1GBu6dMEx+BcobmaQrN1kZrzUPvAqjYQ3EzKSisqwURT4awZUVrfz95ip1+hw9NvzzQxcv1pu/NIt4qaV324pMX4md/k59idnDv8DqmvwTxWljVeqgYa/Z9YuV5FB3lMTW1+ZcQRIaZH9Q6MB5ZmD3x9MPQIjomBRF9O50FOGGiB+rke/M4u7Y+z97YYlRemEAAA==",
    "src/gen_eval_bench.py": "H4sIAMlDR2oC/8V97XbbyLHgfz9Fr5I5Ij0kBRCwbNNRcmVa/khsSytq7JmMfTEg0CRhgQCMBvUxHp+zv/IAOfch9j32TfIkW1XdIACyAWpy4XMnMxEJFhqo6u76ruo//K+DlUgPpkF0wKMrltxmiziy7u3t7b3gEU/djLPxxXPGr9xw5WZBHLEpj7zF0k0vme9mruAZ+9f/+S/29mhoGKzzwGAJT5kH983j9LY7gIHu/YG9Pz5/++rtixE753M5bBDNWbYIBJsFIWfxFU+v0yDjggY9wMcdeNnMoYcNPok4Cgcwzks38vveCl/Lhzt56AvWEaspvMel6DG4P3M8eCf4zDNv0GVBBE/h7K+T07ev2XUQhvD6LIxFhqNdxCxJueDpFUeoZY+l+etxuDOLmQtDLhP5jvBotnSjlRuGt2zJ0zkfwRiMra5YuoqYSL0DuNnBV1evndyyfj9eZckqYwfZMjmI+HUZI7pdUZz1PbY3GAz2GPuDHJ3leLFZGi8ZYJ3yKJPkl28Hw90LlkmcZsxN54mbCp5/X7hiEQbT/Cs+Lv8ci3s0YOJmCMLU5TP4eg/eaOxGcRR4bsjErQDsgUTxMsnEiJmVqb03+WninJ2fvjm7mLAj9uUe4ML2kutob8T2fopX8EpIM8Zv4K6MFhH8yJLQveXpgB1Hbnj7K9GdweJz09uDq1WItJ8GYZDdErlhcpIY/l4H2YIgZ6E7Z3FKn6/j9BJXETwgjIOMebHPB3s9+R4pv6p/D/iRA60Yj+ZBxGveBtDOYi8O1Yt4uETX77B+jpfeJlmsfxT95uLIsNzYJA7lMmNekCx4erAEgiNxpyEQufEh13yqfwL8wASHlSEJpp70PIDBcJDfQdCcmIl7G8auj4/+iqvh6OiInb1/y7wFLHugF2xQ2ONdvH4Prx+xn+kd/8BObgJBu9p8QFe+7AU+vjVMet8wzL0e2/OD2SzwVmF2iz9wV9ziVVzjTnabcLyI7+Pwmyx1PWQ2igIwCi1ChDhWE4RLHPangOULm3O6ms2AdEi/WRhf5/v+FyRBp/sLm60iGnDAXmVABRf4xqHNprfIcdROt4Z9/L4eSo4gMte7HLALHOw6qIwFkwe3imK+CIooinwN/n6Z2t6lQe9kXDumYX4drDHCKfSAjeVYb8Pufe1pKDlskZKzOIVVCDimOHHV5cJxPoXECe6Zp+6S+TGQi7CedVbAN50gAt7WrSIOAxo3xiPDdg3b0NBkZqRLO3PgoWY0d1LL9puoooHW08XapsuS+8Fq+W9Q5iV3k8paclkUZ7yfubRJ3CRhK4GflrArYu9glnI+YGcADwQF4JkrsineluHqYbC81gKOOY68yVnE8aWGPAvLTpyZDQOYkWNnGayKJvrowPUEslsk0A+C991ZxtM+oj5ihBFKTW+xii7ZcY/hdRZkveKnp7gyhLsEBub7wIhARhdUmfksiWFdITPeIsnKnjmEJyBiNa6WDUg9IR60SIiUZ0MQot6I+beRu0S5CdpBGESXoKBIRtVjb38EUeMCn/d7IKWiLI2Bl/rrBTYAmRvm4razT7qYWOx32VXgMhxcQ5LUgueaU88RCysMm0iyAaknyeE2SRZu6v8bBJnQivdcYtHT28QVoDmE3F1fQ6QqfKeHDDQqrQWg6SqN8lWiQd6zIzu9dWB0WwgntOzGHaKB1hPhYWtEOD89Q4kZRCOY+x47nrw+70lJoqgAU44/w5pPWOoHTxBnNnf9OUd5EjOvWBC6yTcSx1vYsOGXsPGttHH6N2H1uD+6o1RBJcvJVXgt7u9pDmHULI5DsdbOYMo3hfSAPZWSHJbC/oDUv/1eDmQNpXTuMZK6hVR5bD4+1OO79UzScnKxnsB6Qpb9PUusYacYq7seK+XwYNCu5SSTigwKq1KP7z9hCWg7gKcHi7KTv2/3Sa4ywY/T/eP9+/DiW48AoIHgkQ98gXcUfFc/EY9bE+9yJ66lGO47XF4dENfPcTUBReGVWDwrTcjWUptbmXBM4awiv3mdVQG1uJlGi4x3suBhiKsRxPMnLvUx+Bc5CHJaWAOgF8uPcuN1ohg2IyD/iuCZWA8A+w01Qok4mnoaOhDz9AzfApXnk+VlTaTYgtVTw2yN3aAQ8kNglmhiSJYDyIKCNluFOVeBSY/RqEw5iWMkF1iTsPmq3KYkfmqkDjzIEkZ4ZaGM3SV4qsB6OgxbXBXIaStSB8QsStBcmEhTuiJ+SPrCk1FFmbqCg+qC9pMiR5fMpjVRYLfMBM90QskWYXpXkbQFq6eM1doKIYUWVA9YIYFPsz9icyKMVEaVQei53gJW/3gNqBQ6kauwIQyDmp2yIPF5V7xOifWMCObe9O3M2qnBlmH1xLBbY4y4Xa5RNkvJsjbnInQL0Gbg/oCd5oyzqo4gJT6tlgn+pftrtsk1SFxui9tdO2QNp8f67rrqXSXzGfmbKnJ5Yz/k0qFsxYHCnxt++9/dgG4K6OP+6rHvIiIFjl2DasXIzKUlrR8Uk0jXLojM7yK9IC7EayFU8Z3wnun+dw//KPYbxesf2Ft+zawHJMwLN4aG1oftiV7plEB7Zl5SeEZsCQMHCVgI2XXMzMP+FGYAPascvQhApLW0BpIaPcXJyORkIvgVdyP3dEajGWXW3Eod48oi/0HTqtuC1a+8h61R43Q2AxWsH0e8RIlohb5YVMxw6WRgAACqWa79TWPQI1A19OI0XSWgFLv+J9dD/yfRgVy/GjoYsxnwVceILAcf0ESGTVA9FR61KJyexStUR8gyrvhaRnQNWJHkPbA4Ao+jNkPW8touPlFbdsOdgPfWORN8Ax5pAYhlOQvDaBRKW7B6ijxu088Sg5KCGijoJh5oLgVaGRhG0pdA803com8qU1HJLPTapbCDUrIhaq3FhbESlmPMnJmRes2SqAqpxX9otCeUc+wnSQBIj5iXujPgyKC1rD1Ikga4FFCJLblVyoSQEkooRfcOlBCJmZrZnUihQPW0aE+FlaoHrnURR8DwKmtB/la4iHDm6SN+yFxQazPW4YP5oFfeDl1FpLUDiraVhiyZZ3sL0FANUxiNGm0VUE+S9tyzL04vCiKMCtFQipW9COOpG7JTUkrZhTR8SKtHsS7lNQa1/CBFEwlDbvFas9VZe0ZGYuEaprxxp1QB9ZRo0yF75qZZAJien7w+Px2VIwBJ+ZceEzFDugFCuSUoNTm5kpJVJv4Dfyd+ch00OZkSO81MO3RSK0xjBxBuosc2sJ4mbfpgpX2fBFdxRhtGeRZZOk3I2M8XjNoGoK1d8SfoZ8Ldg3cpfoEGYOGUlAJYZwGja9lJzCuY+l32zRasnhoPWmMfp+fvWSdOeNRHS75Ps90tTPyRchKUjX5St+AOid4gu8l6KjCE7FVtMNwumR+vMp2ukV47a1O/UdOoAOpJ0Z7ueQ5qEk6oH0gz7+J0fHH6A8zrHAMZMpLtoQ8rVyZPgQjqOnr05WXUxgQQYLrCiDNwE47+BJ2tY3sWWm++mZnN3HMDUk+Hh20GKUCQBMgcQIPwQc1OA+QHI7Xs8++Il5/juL5BekhwX7gBLJGKDwHtHl2UIjIjM3PMHXujDKanwaP2NYzXgNwoV6aZQL0BtYvppVakkgZCmkdvU8X4XQpXaKR307cIUE+M9vyh43i5RGfO2l84YqtIuDjjuD4olCmXvV8Vk8phCGYmqBFrbrGvC0wYy6Ud+Y6500e4AanF3WrTX3oSXQVpHC3RhFovfRmf1OsWXN6wXiCgUKyQI7x+5pyBsD09fqYhgBVdOVd2eoe45QakngDt6ZfPBRgUnefI5SYZsPklO00DoAXM9ZkMbi9BiHaLPeK8OnXCQGQO+kZRzUyD+Xwjw+BKKRjbIW5hJE5oiuxOBrkGWk+O9nTLpxvpEqQPvf1xVBuFkxDruNbuGFV042DoaWoL02tCvgKnR7tNRfI04sojjsiMpK83Li4CasTrST3AGIJgyvfbk24/qUqH5DfPQ0wl9Sn1A52qEFnO3PbnVobINuoKVUg9QezWtsVrNJ5YkapQrP9w/cvMP5hern18bIGu5JIztJAIeWBBQ4DQTueWc7eMhi1YPREetBwom+q2hIzWgsGRrjBJAb0URWwbAwbqr8wGyQJkIkwEPopPN4p42BC9hiGt3T4JDbSeHoct7pJVhBkMW2tCOiNgAfg8pblfoqKQpPzKIS8NStfCPSdImDI1FmmVW6F+WltaXSo0o0tK5WjWpQowPVUetq9LnaZuNC9pU1VXVW8tKYA1Fo4qD2QOrpNIxCkKHaVdETmeT07PmtQoI7Wj+d0UKQWqp8Wj1nbM84pLXzkokDylYECJMZCHm8l0WumtqVceVb4XORYcO03vkBtWwOrxbtNjic6DRfAJtgVhTkEMcrOMNpLpKk5aCVJ4HqTUVNNPcjSPrTRFX5PUBBhnYX7awUGrgFqi2O25MS+ke26R7w9B3kz2Pa1r3CZLwImXjIq1O1MSg0xPIZUytafQ9kIPFlnqKIXrvXakPIE2PTXu4LkrAeup0l6a6jmpT/0s7pM+MXVF4JUU7e2gIoGto85yZRW22XphNOWC7VS2NiD1NGgzHg8WVwIrYYnJ1GhDkc4lNchSttP6Z+XAFqXsDbVJ8pDr2oGjoQNYVImTpEYGFtXuvDgNtJ4eVvsi5CSIFjz95KaFFNFIUemY24zET4FE1/D8HlAwQFd5HphP8LME0llmZrSw0k9gceWh9kbrbBtaT5z2ovIyN6kSAS0YqHCvQGievzpD3nFoY9x0wFSUVYXz65KXDu2pCcq0DLXvzGCqQutxftB2pvGMo9a4WAWwHNwUs3bAvnBT0jOkvh26t/FKWRsC3juAZ+eBDWSWuF5E7sRSCshWSrdO2lrRHBb/ytypam1A6ilz2P5WeRpnHkiOQo5sRookUdaKVSntGiVIoYTIDAVNir4BwuHS2kmACpwe/YdtJxCAVPSS23LUPJahceW8pezZGpmideHjcHfySWyC6hFuMzr+TibrrZWrgjWOv/9euVrKEjGParGrIM1Wbljk8aBCpcH+KrMxvr1bfaoC6jF/3F7CdDCXE9ePc49UUnikWGcCMrMrtWuRg7JZipkBKA9RmQb9MeXzAFQHMMtQfKD4vOLN2qRAt4sw58DrYMTGtVCF1BLkgdF6okR/xjNvwQBMGhdpXyQY21CGOqVsUki3hzVQMVU3wSUZxFBRD7TCsWaJLJCG/AgLnnSX3AiCQwJ8zIujzk/eaYuj8HpeHJVTCkR8q6VQKn2aWIFQDmz+GfaCYLgYUU3yc3vkx9PzPGvPuLGHGlrcGKnjWxijalQNymCbS0EiOGybDXoAcbnkpDkiMCyF6W1//QVFQ+KmKA907pYUdvHS2q0cVwH1qLXpijyOsqDv8+lqPi9sSYTnnbOL8+PxiUP//+YEdu9TSgRDd1PiwhKkNL3tDNQoM2FqYEAnsQGqeR410Hqc24xZn7x+rhxrK5HFS9y4CClU0qFIQE9hrpfGgOwAK4GJnQ2mKM7GuQIQYx67YmKgvJtGhFkq4cNGHlYB1CP6oNWEBa+oAFLuwR/OfhywH6KEbBxAy5VFpzr3V3IjE+lharzLHSVPm7B65NpTzV6/fveGxdPZSijls+wDZ6SJwCCY0kfL9BnPYWnqyOFeUy1pAOADz6aJmoV2llk7yr30N+gJ0J4n8G/8ds5LuaQjsN09HoGAvuTk7CJ7zackjGXsr0IX9VAg0pJngTdg56r8GOtG3XAe008aclxat3MrAmMElDDRTIhNUD0J2nMAonq4TEqMl3JdU48nOeIhcTJE1vWbKmQBDvMC5ICNYYEKoB7Bx98qWxoWrs+pnhsFqVzxKU+47GOA074/vni+P2AnEUFxfyTLlwZYcLDgN519wzWG5tQcGpbplj+DcqZFGp+jnrneZJVH6tOl8V2wEglf5wnj+evgtbu+z5P8ucVtnZ85+0986M8B+47Bqu/A5+5HMrGCHsPECXjWakmtEzrrp3Y/drUT1Wol0PMU9ltfZLdoPcTxpfK+ytxppRwVFf4IQa8NymJYtyhnqenblATbvO0qcHpM2wthF4H3NZpgH5PbMDd9SngKeIosfkmp3nB74/lOklqhYfs7M4M3QfWYtqf2YbGBVFsFshjK2iiLDlxtUrLqjVwTbnRm5q6StQ3IjYKBR9gmoq5gQGJstafJu4If2n3YOKSzbyL5jMtKt0zbYkBYh7ZjLaeWv0N13wTVz2N7Lr3z0wvTUu0u0HFH751XmSicStqcttI1M627WCVVQD1ibep352M7Z65r7lzYXMAcCUESGA3CL/VsfGcE24FeBVCPXpsB4Ke53gomrVLVsaMM8hzknUrHoVh/WZtBmFKrjW1bjIZyFrZYoK7SaI1tgOpxbk+pe6e8SEsXzStOYk+Vl4Jik6ScSjCkUuAHAtPEltMwd0XJxi01s3y1dGA0a1eaaBlMj22LAVy5E+OUEt/YII3J2FLmWE8ZYtQYBSuVgAle4UqII1TjI9Ju3QzbHmXBUpsVavh2Zu82yypweqQffxOTm3TUyasXYGKfAXqRH+L0bpvZ6HMDiVqAbMsbcw6aaaLKORvlTRVSi26L5SWnudnlV40z4lVx4n5eUe6GHyCMUOGUYHa7FrWNbhUjsT+vLNQLfNPbUeG5BatH3WzbbzR2uQDbSwkgYF+zDOf2YSF86lmzZ1vCTh2xMGeZ87A5Q6cCqcdt+I2YM+6/PpYt+2WTfIDdvLwF/arKyiSfIucxBhUzN9IpE5m5tO7iPKrA6TFuL2z6zJWZlwV+o+31y76vc0HAL8oHWghsnSvYnlNVpHIpNOO/CVxDA7vFHDXCoGRwl91oaCui4efzNMBAKVW5K99ThJECFMy6JE0bppFE7KXVWKlcBdQj+6B1Rr0Ws7ejSpA7gPWbrqTFg7M+w+BIWSjLemUsB4btL645T3Qc7MpKQ2DHJo7WyL4qgHrkW2ygw8NZfxnDWLdyxrF8Rc0lSSXsWJYJgCqJ4Dz0W1bMLtABozIp9EteWOHMWRq+s0s1qQLqKdBe8PMZCWjKiMKMkIxT0nrIZ1lhE1EPPwx0zFSxer0uonzcmBO1s3BpC1aP66NvxM3H52NruKVhPy0ln5LjQircAKwNd3jW0LmyZS8FvzngUQXV49pepHPtG2ODDABzbbPkbSrP4zPM+FnyZYy1JhTTK7xh2soDZR05mXWTNVceVCC1WOtKL/47qnZu2LuCLfhNLo9Q8y7r3dR54wobOyIQCOzjyfjVK217jRvHFp5pNvfVyIH0KJrfaBFP3lww6kSTStspwpZoWL1O90vtI3WpcyKlOiGojjktH1LyEajMsBsbuVMVUo/tsLVlfBaHt7Aqk0XgKWY0WseZgTGzGPtMoHaJXTbAmiyz4AmwM8zcoVadiKkbhNriUwOeYcAzmkOWZTA92lb7KSmud6k0Tk81zqFaY0pZwGIpjEJjb1O+RFzBdoLXxoSUQPrRG7w9nh16jthVH1CC0uNst7+wCXcUQ2t3CO1oStCIZCUt9qvwqYPQoX0A+xcLx3y4gywqivQkC3fKsxq3iLF0gDEZvhnNd7lFyqB6CrRXWPsGG5b0qWdvSX0ekcb5r3/8U+FLH6W3Dz9JEmFDTB4yule3xrEXiumE9m2zl7YMpse2vdpZbZEcRRBG8t2l/p0nUsBS5tFVZ39yMj4/uXD+dvKTNtMmL3UTlpda2V2K4hSkHt2H7S/vvOm1Uqd8dHYtgwgb+sqGlmRQyAQTaUqsewzD1g6W7lzbBArtBTD6zaW9o3yhAqjHur2i2DHyZpU0pgp9R1p3X8XEihP8WeQuwHdvGF9K3qf3+Fnw6y4FrAymx/px27lC5PMT6wCD0iV9WcUUpz7aTmWPbu6t1ySdy1Az+SxFc9J5BVKLqG18Oy8fmhHGzdBnHVX3fhXEIU17JbEmj6nxG4xio7Cq9/mheTD07+DyqwLqUW8vQljWwihL49V5P4TJDKu+oXMudTCwoHNPQVaTUh+GsEbNNE8saAwSboDqkR22q1eXstoE/7xCceuGqq13x+iZvWHPGgwG3eqihlEjgeU6Wpv4s6NS3Jot4gJMj6n1jdTrdbFEXn8njam83Ymql6h6CnJngDKnZIW3TiQnshyCPHuNQrkCqCeA3fa63lbCYInT3JuHm/y5EqKhHndkhq2fUh+cwlV8eIfIlITTo/6gda17NQVZnK1kfnMloFpSM5fKPzZFgU3bwbtd2xraNAfr8wpUyVvHjuywOdGhAqnH+lsFICevXkxOXrzL+XHesCaIVlX3Fi1xFa+paWcrzPmVg30XwF7MzGhl7YjXbADr8W4vCElZgyn6st1Chmnd12W9XDEE/aqW6rOqc2pWRzZB9dg+alslwS5OElNxu5xi/2ZZFqhUlGfvj8+fw4dZXPgQSGopaJ3H79pOZw7+boSN4rkKqMf38Te1Jp9zTNsP1Z4u0ndSrPIoShqK/DNtEY8pMit0PDNZNNtSG5BahB8Y30QfeRUhmpMX2ArfC90rXnBlEMxYz6S0k8oc57BKcPmrpc6TL+Y3aBGH9lXzfi6BVXL4x+c/nV2catP41U+bmfzy2JdWk/lrgpXWgI3pEkr5EdsPYv/Tl9niarFypovZ5WL+VduIhgZzaJTm9gIVwM0FsUazPbUNHQcKR6m2AWcLuWzRTmkzJW93TZFC0QuB7ypUKIPWItempnY+OZZ48SOrx5b/abE/MfLrLhNEy1tNcW/Hum2cCtsRSzsMHesmMSKr2empga5FsE2H2PHJhJ2Mn5KO3Xezvtun+FMMd2ArnDzpSZ4rpPJo0K/vsprwjG15juVNsXWBFzau1Q3IWnRbrc/EjCZkCjilNxmPyI0pCwyJV2EL/csIfb41MdbQwpthljIrEjtapG3B1uLYXrwRV+z7AB1A+2Ld8cPHZfufxmD4QIZSc30LQ8tunmq6heq1aUVW3tLJgRXahOs2cC2y7SlYJ+Nnr/OuP5dsHlyBqXQG3P3yBTUlwH4deNAaBiSa+ZDl+eFdEiUqcLUIPmo5kXTE/r68WfzdyoKffnwbvv10Pvv7i3efpsPz8O/PHz/WJZHSbY5PPrDmHNIqZC1Kj1stppFHd0gmk5f3woSNn46xwoJr23f66CC/A1vZgKzDqNWcddx0mNkMOFCFyKrYetxEycGHRw/l1pt7fk2nOSPCmDzc/GBXp7kSZC16ZqscZZrGru9hR281Wx3gpJnrd0dSMmYLbGScrKZh4KHcx/qu8wtM4mmQkAtbZLYPQt2wfQ8+7/ApV2FrEW/xQLVsiuJCqTcd91//+K9fu3mymU70ZVP0fO/W26uAtZhYrZe/POfU02YGKMdp3hUTFffoyDTUPw/zD49ZJ+mxz8wLY8Frqlq0A5LChCHDQSA+p5m+sEWdw4NgT/SPf8Lco2KUTtT93nzCrhfYXuYiXWEWzfDIve/2oydsWgacDrtPWDBj0/vTo6PpcCRTTzpuf9pj7vdT+HGacvcShv/+yNyoEngoT3mM6woF1nPTXirYmzgliYx5QfuDwYANBn3Wh7/9fp8d4De4hN/xcn+wr/EJGKm4U1Z9FbB23bXn7noXwKL8f/835RUbgeqrMLpW6Jfe2ibShV/MOeoW1q6stgpcLXaH38I8kKoGsELKgVhFeDoVaFdLLgQF057J3gZrbrhWObVmgwED2LuqP6qAtei2GV6sq5WTAdP9smmbK9mWIQMUmD6ShJgmop9jtO5SK7HsbNcsb0DWIv6oVQEoFy4ZBhvMDlhJxPF8oqRvwjqAyV2wzlkchvAwUMHhalefERLaqe8kztKMVsLckRZSha3Fub344kt+s86JGLHDw0Pv0Dx8+HB6+MiyHj56MDu0LevQsgz6az/0G8QiJi7dgTmVweoQHBotm76ocU6DrD8LA/K1j5hM1WSv3jEpBbBnzyxI8SyTMMZG2g2L2MNeXBgvc2ah2Vj9ugFZi2+7Wtw4ThKeiiWd9aT0U2XsBvMFEkJgDWIJRcq7BVA8wCTl/qo22GIkiZUKc77Y3RB0C7YW+2HbvuliRVMqXn5WD+AsQy+1a3hqUtvOnUl6FbhavNr0UD2DVYw8CDnxiNgx/AYThcFCCiTR0d+rqVTKS2fO1DugfUs415Z9uYsTV+BqkW2zuy0tRJWtVpiNby/Ofxitq7ZA27typfdROp0Lq0TbyTbLTM9yoixdNXexLcHV4vpNA4UjBhbIP80em8KfYY958Aej4Xn1C9VrA6Iycq51c9jhzHYAbrrDPimB1eL6DfWofArhxeIIk7d81nkKnxb9Z6t0xnOXQVfb/gvAHB/ALKu5/VcJrhbHFj1VIUgYzBUldxQLIsoIV99yXqwaEpNz2V0fgUmH7UonlsZv5ZgRpXw7npE2x0u2YGvRbtF/lbpBSKfY5IxXeprABJM/rLv+UOfIumPXU9sMwzuZOJugtTg+brl6eBq4YpRrCtRLOnDl0QPY3ZEZBe9FTizboyJITRnx1LTFHZprb4LWoWsZrSoS+Sk9yGNxk4oVnkchVYk3k6ckUUEllH5omXoZ1cVI8rN2dgibTdBaTNsL4Z0s40+BMk1Vwmgp8Z/TrwWzfeMmTTn+1tL4ZN7BN1SBq0Vy2K4e3H8xfsOiOCItbyVgLrFMuDDKVQyfzq8jsAFlDJdqpdc6I/6g693pLZ3IiECCptaqOfNsE7SWCO2VJE5ePXupFrNS+adpfMmjQqsQoC3xbEupgI0ANMn0GdHC9BcO/R+OZjU35duErcXabjW4YA01vFler80Vxb4Twzsx4wpgLUYPWtYk6FTOPigR2GMuwpNG73++nw7YczLcVfcu5bnMizoSN0hrGFSeR2MurV0Mqgxai+5h2wpxSRVeBD4o/agKTmUm0TTky61VLFew4uGYioOkqDvpQeq9iyi5i3qMYLWIt1dS+BS7/tQx5sSd8x4Wi/au4xS7SCqvs2iy7wzj8g7MuQxWi2ar1YRhfD0LxKJs4pGUzb8zL6STvQAnLmTv0PwnHUuahsb1zBQLstsaMa0A1uLaXjXhKxHPeXSblyJJL3zOhT2VbxFIoLuzYFMYcyu6vYMmtQFZh7H9jSoJMTPu7TGxXsE6x71x70XvgqpvDKNnmD3T6JlmU/DJj+w7VRxV4GqxbLOYcLxw4d+hkbuO6YA20iJk3VFZzaBziOCXsNCSdd6mhQ3/3lWf0EDX4t1eWSE6JVgSh7dRvEQ2q1gwHg4sxdCic9OVsskBaxD9juyzrqY/S1cOFgZGxtJszurdBK3Fsr0qwlMvw4rATQ0ipsvF0Rl1DSK9DLT5OygSVcBavOxv6JKQzGbtkaBSVzrOea34Fs356D58DxQ8eAidzHfoPE/d6BIkU/+cB8CwutocbhtATOz9aGY7sj83QGvJ8qDF5tY+76ujkNZeC3VMEh6jiGwsb5GZkyoolamgDqL3w2XmEpM2QAO2HNitUWSFOxqWbIHX4t/6QfBUcAJWb1DK30chFa6WkcBOhLLXv1/K5U9A3VROHR32MJUiMcTOQzergLUYP/wWG6Ea68fdntA2+MxcDPBilsBax45qdOmZRUf8zGwvM9Jd2nQVuBbZR+1Wx5K3Ag/5WfPukSxqV6e5UOsS9uJ5Z9ity8+/slPTzixnPhvuTNAvw9ai+LjF42SDEA+o8Ph63SqeTXNssbSmhB39ajAjUfPxZiWoOmQetB3Dm/Sn8c26LmZUMF4Uu/TUdX1+PKMkVsrG1+lQ5gzWnBWRk8luLoPcgq2koL8/earNP8frm8nn13zaaub55H+/Lh+Kus9AxzKPzH5f9VCF9wnjufaEFfE5VEeY7uBDVcDNqZYYDVs8fGkW0vPZj5PJ+vDnPwkvDZLsz24I2mPH7P7pQF1AKSSACZNZn2o1yBshQGzOwp1FNFVAPaLtnowOuxB+x+pG5DyDwYH8l2feAc7etezHXD7qFk/cwrZQ2rTJbOGA0MAazmblcQNSj2mbCtZ4cv58xF6cXDCqbRPyOBw8FidDo2/dg8P1Qb2gswfRnNdZAyKdOZlxaUXO0hRiV8eFbWg9rm16pSaEqzwAY5FlyejgwDx8PBg+sAfq70GI9frZAViybh+byBzotie+uxmBsrejvq8KqMevPTeUaqqdBN4lKEc+FzwNSrX4XurOUAm+DWPXzyv8MP1XDNT5aPuVg5i7+pC0BXIyNH+1nMSE51g7AtOb0HoatNjkihK8sKgJTxkm7WiaxteCswM84WuVHOzqlu+bqYWKzi2dHbzLit8C1iPYnm40mVy8Qsb6V+C+7nDEvnzBTm3BfACWzFJ0ul+/kgUgG7gFc+3qzUznE0gNu3nprqH0KLUZuHv17PQ87/bDDtwkOMCDaQ7MoYWzVbqQd7qRzAj3p87Z5BupY/tgmexsTLYJqkX1d6So3zUBWLFawaSopNxf0BcC9sPbV6dvC7WBfMmSUckjPg9IZWAdJAe1IKQzYWuygvOnDMBGUqecyfEnJ69Pxhc5L2hMDM7HeMJS0JgqI3b2Na+236NpOfqyn7/i/oh92NuvPpnm6/n56Rv6JPr9D0DK/Vy4wB37N/tfu9WkYNtgMB11GcFyptrT3tJ4GgOewAjXPfEOpNv+QB1WHWML8WPZ+0LbQjw1pkYmHBhi5zrcBNWvw/Y0uTF25eBsCbpLsgpzASH3XxqH/EhusABbbiGkNtXMuDQtZ2lHO1LsynB6vFrt6RAib8e9JA/3KnVyDSK1vJGL8LwelvoPrc88mt4WX7R+ezPyHdC6AyfbEYfaBNWj3qYm9+OPJ2WDA+vqsEscQxMsu6VjvGARFwqsTh+/sahMDvUVB42qrDkHWAOux7M939df31+g17p0xss6WVQeoU0nIofzEQL18vMqqH9Btkp1K/nTdYZeacuxw7mRmjBiE84aaD3KLbq74pVMSPIpK0n17XTz7lmwS19evHmd/6RL/c2WIR7Qu7Tu0KxzG1iPX5vOrfHp+YQtAyGVllWqOJLkrv2x7EDcPw7D+Lp/mgYgZ0bsPp3UeQ2aDp5rhcf8oWQkvLMFn2kTg41UoLnhGdHMbDZPqpB6CrSo1fH0Ck/iwyPuQY9L0AopaQGA18V1MEdtz6HGMzy6GuRnE/4Q+RwUWu4/D0L4OgaqoKrb2UcVfx8UwjodMLtupkEBpEe/xSz3i4uzfPfKhf1j/yy+Bm7t95+WmtJihBEUhEvsbSD0LtyFZeOhXWg+7/CdbEBqcWw10f09n05i7xLU1/JpnLR4BSqFxe94AmWGB+lIMazKV7S1wdZUGN6lld3hAM4tWD3KLea6u8S1Ij/IV/HUDV0SwNQqSnrrI2+VUnxmrRT78qBKkXCte5AOrIKN6ZP33Znaob3THaq9Q49/e1rW2cszhtDs02o+D2m69w0OVs0+OzrCj/aDw31EK0tXnHXCOBa81ItcW6+ySPAFLIeG3GGdbgPrMW5T/3qRusmCPJ/AsvFMZcnJYWbBJncc4WED07UNrvJkyIjRLe95aicL8nFmqSGS3Q5R/Q16tNvLiD9L4yymmU7iMFRZ4o6DHYRixxkE4pj28ZGcaPQb/nUCxtCSp9oOj9gryKCJw5Kj1S6cdeB6jNvLiz+Fvbk+K3jEDvKPf1ml4RGahQLsQn4VhAMPewplxL/dFZ1oqM2bMBI6aE86VJpPvSgD6vFsMyd+W98sqidlk3ya7fPJ8MHhv/7xz5f4V+W2FRUPmA/18s3xWGV+1Sihl9YtcikYcseMbwPr6dBe3vyztxOY7insWWJj8iuF4yhZxBw+HBjwP1P1FJf514qh9xj6XREsb1wNZEivAk+fP4OO/inYTjsTaMqAevzbS6B/qRhVnC4ltyp0b8XD1DnBsgMVNmwm5kdZcEyQAq/TVkzfh+U8M62wuSivDKfHtU1/3Nt4I3j15cPeH+fZhz105XzY+1pEsN7E0Tx+9pS2tqxyqgmtR0YRq2pMninDaTFtMYd+cgxm04/w39o8ZNepqyoQVXgL8UxJf5FLu8mSFDaYTtcggpIdi7cKqEezPV/W8dkrYkKwWH84fz0ih+pf4MKR5EboZg3j+Vymv6lurnBBJ5DtxCTOA2x+pyG5BavHs80cejIk8jiWWK7VrvHr/sUJhkRuI49NeXbNcTOnYFf2OR0ZjC3Gvcu+XuFEeebI0XbM6wakHuH28uWByXAPV+1GsIcCBO6VO2KnU1zE1D1+ItP40A0kr3Z0CuYn+wrPxJOBm0YZVAHUY2q3fHgJNqVhPMKGrMCAD1R7wwU6AaiBPqPTcYtGh7XHliwN37rjqSUFqB7HVrPmtwym3CWO0pZnI3UyUya/qggtZeHK7K86aymFecpkuHWnnVSG1aN82LbfA/PZSuZvSfAoJpxHLYEiBZgyixsaSl+Lu6VRVOD0OLeYNE9xSMoWGOVRycGvQULdwNwgEkpdYEU6UHEEZl1lno2jgH4Q7l7WW7B6hNtMnx+fv35entXvDP87A89nwcNK0mmQpfLsIZHEeN639ARpT+ExZ3fQIcpgeuTay5d/g95l+A+0AlT85Ila1Ama5WYf6YuF1bdKfCqqjfwkDrTu2qUthAP/mTDmjn6Fm6BadFtMln9VVKlgM2jMNqSaQ56meFwJbsfcW4dhQAy8MGnz62rx0tRIaQ2iX8BaNgakt4H1uLaZMj9ZTf14CXuSZe4lx5nFsEoS4FEV47fHb04onRy4M80kWUIrbK0awBMbbBwBwxpL24yczL608By0Rl1xG1qPeXtJ86/juT1Z8DBknfG7k/7QGJp92x4OH3VH7I9fPoF8GoW+m5RM/AMuq2q/kpIJqPeP5/pYRGjMbbGwwtCBcRqbOGxA6pFuL4d+DLb65dovG5CzWZ2PdwuKBfwoMKKYyvNbPYTWS1wvNGEg4LK7UqDKcHrs2gwPrpVFDLbUxFbGsLJlxqYYSWcVyZ/rIPQ9LJ+WQZa6yIpnpb5wrk0AtlN/Z3hlA1xPgfYCh2PXw4NVYnStlg28VQSmEOzXInooYdj42Vvm4U3aM5WABzmJYYpmzaICp8fwsEUleebCEMwrzyKFEEYykABIyfyRRH+0jG/NbBhAzkyzalwG1OPVZnzwJSawqAkq6RFSC17ryGDrXcq22GV4zK0Bs1dn1xkic1QYaLdWoYHWI95eWPD0GD2m23HR3NsKpnVQFN7l8dC47JvtSecruiMb3a82/Hg3/2sVUk+ClkODgN4i9kEILxMuSx7OfijyVSlFjNY3LHw6bkl+Q2atm/VVlpmPnaWVLQwfZCqO2TjvOngt2q2m1KNbajOFg5TJkIMKidbQiE3evZAMGlM/chuplOchFq6vP7Dhau7c3HAHBjLsRk69AVnJrn978p5dHE/+xi5+OjuZUGL9vXc/vH7rvHq2lV1/tQqj35Fej+A8dadBCHg7soNS3swegfFw4nmc0gDJtY5+v/zyi/chuooDSRGH/LOd7pcPEaMUHzZdzX62hh+f4IU5WCIduNCFb18/RHDvh+j9IvAWrPIif2Gd4y57usLCBlIN6EzkztMue05acV5q1xl3UQnqk8ewP8OerJ1nXWrWPy/dqaX68ebSUpQb/k9QDgcr02y2zH4+tCXNhOdGs86Hve8EJu/BL126TF0+Zx31vSAmUEdDy4qfuYaS43i5RA9dCQ6IWfWVaEn5tIaUd493tkVMIt79JMMcSqxjjL3OoY30wbXRgeubnxuJpluAm8sN6PaMwuhsvfzeyNMeMMFCS65xDbnsb0Iu4Jh6ciWL5EP0RxkwPmIf9lTK6H2ZL4pMXbD3L0/OTxgsUARgA/ZH58XJxc/7gb//cRfxfpxMiGBY90Bk2liCQCgMV/0uCj34BnuzmT7cW8SA+p9k1In65x3tl0lBmbcf4fuHvf0/f9h78rv3IdEJyFOthJF87Nnpef2GA/Fwdnwxfgnv8XZLDNABWr9DDhB8JaP6LpvunCchsodf1nz9FyknXSbcGYZySa5mATbzOkvjK8q7QiG5whpjOuSrOCZGLx1Jz5zhA9Y/T7EgF1QE9MoKgPl5j56P7wzszEtu5UfFIfc+FnnXn1foUyjfOStuDX7lMYJvrL2clHePn/x7xHwe3ABtUHOocObKIhqxXxRWuD+lrAWai0W8CsFqXGW4SpH04W0DNeUQyn1fflgDhbcfi1ipq3Shmc4K8sNeLX2tlpaqfkOvqbsRcf3lv8EBf2kiMZcnra5PgG+gbTHwLhrSoHKJwvY/+fHs9emrC+fi/Hh8ssUCKEajZwE161b5lhy6826L9gJJqu5b20LkoLsOIgx8vXfDS2yXH6/mC3UkJ0/EiNhkQoVUKJCxvcl1xFQtxP0PUYKtItIYXSed/cEBboL9LlxWuvcRm+4f79+3hux7lljDjnFjPAJryHx8iEADwSMfO+J0FHxXcWVsSYdeRmqBhC8y2NuiN70fkXtaFf9IEBlEBdSKiRKrKRKRbpGX5CSgYKD6DlUS75TH8dBxneLiBSNz80G/yRjb+mvu6aZH82C+QMobA0vto9rnzWZgtW8/DYiGDft+k78PgIi/WcPB/QK89IzhjmcgJbYeABc/dD50f7tGM0rx9t/ytQEfslUageno46nad8OJy3lzaA1uPg4u/gb/ZXEcit/UmqlDY5P35Hvkri7df3eHnJ+eoWKPS0+OALYzdn5x09u+Ov8OT7Glrr2ycyLAVjfJxtI/tPG0CRoBPkz3n+7ffwQf0jhx6EElPWRfsIWbJBy9cX9pWvByOESrvFZhyH9jtRdjVWdLXldLXH1RaEiF+fet8X93TwH56JjV37nekRZbj4KLalUP4jQASO7/Nnf9Oc9+3+qWyT4aVOj6b3gKPU+vfrvkwLfuF/StW+cgIO75fJY3k3LwgKeOPIYNpH2X9f+Mf0d0w97eXn7G1+TlcR9T6xAcGwhQrxf07VHYdBlEMihFx4oP4D66X21qvCcMpgOxcGEIethAdnzqdLuDBb/xgzkXWaf788g8zN9vugpC35kC3GLppped7vqVnuIvUldc/yy1FtgzVzyiFGeV05q/CRh9TqkTAQjFj3Q936jO0kXZ8oV264idvX/bo5MD4fP5ybte3rcBvsqz83pSoRhhi4Ov91SZ3LHvMwpX5aNuVssRvdzMwSnu0afSO5Xuw7fJK0pH67VBdy8UXOnOAgL/wdLxI4ZZCh1v0d386eeCNX0EsPxltsFkTbKjGBbCTn6aOGegAZ1dTH7O7/u4fSPwwABZiiNjNwktMPmw8orzFj/n3PBj9S2rUzUgFuV3YOxumc4Rv2bIcijDu6AusWWsPUYy/ay8YT22toh6rKIdfdRRdz3G7yNsI8UGsO07G/SHNfRTvKK2MS5JAA5KzvjiOTZTuuXpYK/7P0DdXA5XwNSWxLWdL0jQODEYdwSvP6fPg+N0vkKN9ox+6fhcFtcCVzjaeyHVck7ocbSXJbtY71+Fqxx1AFqA46rhOnv9vrRggKv5MsRy9BaLyyp4lf9Z8DA52juVZk+CBnRH3TiisPgBvsGBl80kexl8EnEUdtUrwHORP6g3oT/4LrATJYGCGYEM5DsVS0R+d+hxR2S2d0pwcmweCt54h+Ogg9lxuvhgDODKP+wA1BB48T38gC9PHzYQkIyuNGg+xvISdKyO/CKO8NCgHqw2WN9OfElfFWYV/rjFfiUMsVmMcHRKD0JmuNfFJOuZllvVcKrZgBI1Ovj2AzwEVeDGAm1l70OUrwc6s2i29x6VIfYFRukUo3W/ll8ZbIsvpXf6urdmFxOwtRSHQIvCi8NQmnkityzG8QpTo3OJgPirSx2vwjEJqSpO8kVRSdy4raQ3NtxI7Kt6Y6FaNtwndzfeiUQBFR3v3GRBururhGVsLLELYHbYF8ndgALdr3ubgM9yfEqghLYG9mLNmXNI+qKB/CEKwNLOm0oprOAm9Wn7hnFF30Buhw9R3M/HWb8HW9Qh8eQ4WOmz5zjIuRxnT64/ycbu/X9qpW8SJOAAAA==",
    "src/process_data.py": "H4sIAMlDR2oC/7VYzXLkthG+8ym66INIe4aSDr5MQldtVtpk46ylWHK5XFoVCyIxEiL+mQAlTabmKXJL5ZBny5OkGwBJcIYzK7kSHcQh0N3o36+b8H3fO2OKwWVTpVxKUd7Dpah5LkoOc3hflU+8UZAhieRKgqrgp1LmlXqYp1VRMyXucg7LqimY8n6S7J4vPID2CZq2BNmkx7WRm5CEqF7BfC7KujUSjxv2jAtVq/oVS86zL0kpeHPPx9J63i2ZmjTzfDRVFHWF5vxNVmX3mzX3NWsk796VKLi3bKoC0LqHXNyB3bjEV8/zVLMiEwE0jfo1KzoC+q13/vTuKrn+69kniOG6abnHX1JeK/ioyc6bpmoW23QfWI4qeN7VL1fX55+Syx8vPl1eJ++vP+BeoIn985eaQkFrdc5WvIngquapYLkSXC6gfi5n0PCnGTzzuxmkzapW1Yxiw0spUhmBbwS9y5/ZSiIpQzeAVLye363m9IQ7TuTAX+q8EgrSKuOR74U7el2cffzhjzuqUUZwJZR44oDBuG9YUZCaFzU6Vfwd80Q0Uv1Oi2eiBLZUtG3VuuJp2wi1mrNnhjrg2ZiM5nQPjU6+P//l54sfz67w2BsfjfVn4KO59ECD6WFM1r/Ukh7WDvr51OaaQz7wPCe7/FuUm/ElCJkgeZJWpeKlChR/UaGJD2bM+weePoJYAi0D0aDikoIwb3jOFCbbI189V00mI0owYmu4ahu0rlwFyFpq1iivnnkThBQO0KuuSaFVJTXllrC8ZilLVJWkD0wFeKJq2lSJqlxgvJoZ6KxPSLBdMPluX1LU677CPKVXdJePhssVBrhIMCxFrZICHdDvslY7TT6KOhmRLeCuqvIuO0OYfweZSNXgHIsO77S6FgUIIkhr+9o55Sv4QyvyDFrJGyiwTBEq9AYtdM7Hoxxb9Ta63jFWL+1wfRPD0v9cfi7XA+nG9zr+CcN6OTZW635Bm2b1k/4CbkY79Lf2myrnuOWTEjrbjB645Kq1mR1gZQi1UrFSjflNGDcjxtv+beNZV55xLJyCINqYBcYsuGOEfljUFN/e/J3IQxybEhn8MCLCMOzgkKbEtrBfnq7X14vUEGKlSj6wfQWUjwvI0MZUGZTtEronEjtl25GEi5HzXmfXrhavNsCbzKL9GTSkgBE/jv/oyK0E+o159+acM/m2sZDUdV5qmXmyFDkPTJFRd1zoptihz2jpf4Q3ohzgxo4owODPVxc//AVIGUTZrIPNPciTVq1GlhP9RqfWWCTd+7NQD1DVvHTMCoFJWA7JoCrF8oRGIol8si2CU43jCeH4MjRyFUPgpuEBSegR0b8g9A6fQkITgdW8cHO7GwzG+djT0gk4awT9wsyoGDuKzrCAZBovfZ0Dw6lRyQq+Qfe3pVCxT7S+VXKsqBNT6rC+cQqujpUiP+hhEV0xYUzvwm5q2v4TBF+xnsiwR7JMaqscjUbES6ey6ETinparI4/JoEdG6itIOUmoMc132s4bBe/p2Xt5O6Oje66C0bnoZj+cvZoRo/M2FhPRt/F0sPoart2Sj3eXviBiBxPi3aX9IsI9Ed5G9lEsEQhFiVP6JAUlfPSMYykPdIpmbVHLoI9/CN8ATh7+9MEGeXA6Od3Ztp8E5/qBwafi2qNkh1iTcpwRh2h+H8O3+02tG4TTgCDhZ9aU2K0XPaMu4bVVuFvdLGCN6Rnw8GZxenJyu9lrp+tCHItrqRHWgUH8lhwQ0hsrA/Cff/1jF6MWVp/N8drBtQ26jhV1jlAcrO1Ri+h0uZGh1c5xx3dwMnhjdOA//42mWSu35Buc70RkLae+wumrTdoTbLvX2m21yUw0OLVgwVigx/etJums/D97ZJ47XVISpDHoletao7Z7aI0HW9igflQ84v8Av5pxgJAxfeLOMCo4WyTVo361LOboGD0q1eCP6D6v7gL/60iPFH4YDqokPQc2IP3bSupi97kE+IBuz2DtMGxGpmK0bDR0YXpdl6LdRGQv3deTHh/QL7xs8SsVcc6eOJUxN+uOG6vjtM8Yc/qte3myHoSbNI6iyCkb60V9duz4FI5hi9Ebo0i8fxKjnzNX8uyVUPw6uA3HcxDhkM3831DtFECq9sv+rmYcSP1Y2MWNPfFLFW/rURPbetR3Po6r5N56HIbWnWL6pO+YdksJM6zCsWmyiLw3T1GUm+OMlFWjeHagYsYQvz1ckhhzjih3m4E7sE0SbLU9M45NEfXpcDpYPhUKJsrAqqxv2gjgulu36F1zj+VXqku9E9DQ2gjdEuMezPrbR9LdXj/iRIly6YszdCRHLMsSZkUGvr0e9GkYXrI2x4G3u3fEtQee17H/0Vwg9tB4UFw/Qo3l9TePvdQLe//4OrE6W5GXpcZwiSw8wcGQ9wJNLg5XnDoXD0s1VTw3VYyC0odKIHd8012P2U/2mW03t45ZeuGg9LLaOeCA+h+xKXNlEKe7tKDvUspCe1fbDfYRBGdGjQXupnmbbV1zRF3dozrULKyC+kEqyiAcEnJvU9OjQldhuEcIEBB3pFfDraY3ojDLzikDvK2P4qOvvz3pRqWhhbi37JjGczD9fn2kg39EY4uWbS62aWaFIxvuo21pW4d0Y8/Av93DULE4joGSiJrUkEZ9aeG206bMlblrt+9cpU8SvmYq6Li2Okp8CK5nrjLH4Ou675BwolVTd9HFkvVdpO8fCNyOGce7osafCkNUyXlOk9/jtUEJDSqL/sMbVd9M0l3Ya9v1kGjThFejez6bOjre4ysjh7lLiLLad/l54AjswVfff7y8PD+DAC0Fute1gKurKHQvDbooHhiC3fF3cjzZtWVyPJm0yKmCN48je+v1WhtFHOjp0dixvxg9D52eJDS+JYm+FU0San9JYu9FTS/0/gtBDjfvdhsAAA==",
    "src/synthetic_rev_pwn.py": "...",
    "src/train.py": "H4sIAMlDR2oC/9Uby3LbyPGur5jAlRKYpSiSopWNKtgq2ys5m5IfZcubbHlVWBAYkliBADwD6LEqVeWac3LMNT+2X5LunsFg8CBl2U5VwoMEzHT39PR7HnAcZ+ddGi9iHrGTOOV7RZnG6ZK9DUWcF2yRCfbs7GT/WRZh64ss4onceRpIAM9Sli0WcRgHCXuXyiQrVizNCj7Psgs52nkngyU/2mEszJJgzkSZsr29ZV6ysxmTItwvRBCno/wGWtdIli35eh3MHoDw4YqnB493HJhCvM4zUbBALPNASF69F/Ga7yxEtmZ5UKySeM50x2t43TFQmQhX1ctNsE4UShQUMM9CVjhJFkS+btzZ2Yn4QjWFWbqIly7x5F/wmyMmCzFkqtmPYkENzGOOapLOgO19w6I4LFA8jMEEToEQU7NSQIxY+OHJi1O2iBOOc0RQTRRnAwRxFm49zoDts4Vzaxi5G+FkFCL9iRc2gRG/jmUh3YHiAn9XMdDNcp66FtyABZItaiD8CV6UIiVhjWSw4D5Kwl0M6rEesZMgSeZBeAHyZWtQnR6bmCIQbNTSqybjWDDOoGLaAtzKtAXXx/SiTJJ6vH7eWxO0UEZLXrgOCVc6Q3Z7N6AWI21qqucPxio5uFTCX2bFSVam0bEQmXAXzsvM6BibyceIyhGzdAezVza2CqQfSAmzDtLC5dd65mu5lDALfk1c7K65RH+TuwMG5N6f71iTAIdMXIQHocB45vX93uRcYQM/BU+L3UE1KPlax6TR+H00R/2elUVeFsbEh4znWbiSRyxOC+DtYAgOIgJfVA0vs5TrtiDJV4Hdrqdl1NPrWkq+6tVAdtTjVM6nwGkuEL36Mapep6W/R+zVJRcijjg7zd48gQAigrVEa8xFdgnNEXNLDIPzG1ZcZXugniU3YxnTVQJgscTQSBOtbdKex/tdsXtO00b4BjaJ6mMo1NA1KXqtZ1VkRZD4wKtAuWN4HOEf15p4LkApYKY/pre73u7vHo/vtCNWHYyd6Vla9joq85wLd9AFftEw7YrXNFjz3fMe6OCaSf6BJTxdFqsO2jq49qHbV919BEhZIkgvOriiC96aoVb824LnbLL/+IjVYdlCA8kw9h76z6kf5UAgo9FI05FAYLuQH7F3GB0CWXwfyzhLSUgUCZ5jFmQzRVIOCeY0SJclmFcNBbmWC2nio9ECDSoxIroO5VPHCpSUUkqdqXVSa7HQtKwh2noIkSXD9NUCHSE5P4cYg9bAI7cRPxVLqGWvoQUHm5zzYQOYfB1C92weFw1w5aJ2N7jpmSj5oEkAHNFfCtAFBDE/XPHwIs9AVaAaz9ETdmqMOtIX2QVP4184Ts9MdWRaCQ60wO+XYUNFbSnaw3SAv6Acm97RQmh2fnEVRMVNzj2K8C1BtxyUsV//9Q8VFWhUDKS3lpOwPct/jkaTxZ0cNP3z+zdPXrBlUEAaqrxlMn3KUs4jyX79+7+n4+dPmXs6238yGY8HQ6wfJ4fQdBUnCXv16kXHbZgHxZkqP/cm07lDaZKqwlFYRsEohgx8GcRQlSbcrjwuISX4yzk6eQ0M8vIjfhmHHJQKZYkoYi7d8WCkYu+arzNxA5XahP/BEIIql3S8kRJ2Ag2DANxXg/+RTcet4oxKjzdligJVVUejn+y4iu+H47sfU6ennzWEK/iHMha8li8pwb2tOL+DcKs5Ip09fzoYbaKLoe90xtxnVOe/FhnVLKgsim3FCnKd9pxXbzYRAZdvrRyYezx7OiASoPEqE2/gopp7s6+bBqaYBt7ygpU5C1cBLBb4Ok/A9joJYYoJASAx6nSg69zQF0BQ0QjuN4g/MLrbobpD0DW9Q23pezPnc+KgjdEdzfTCaNhDxXxvHHhmS0kXZqXgUQWv1XCAanhW9VKS7yjgABVQwaASqBQIoiAvIFfWGvgvJsyeLInCyfmi8AmgJ7g3w+gCUgAswbl/STT8JLgB5j1YS0konUEcsBqnOhwLQvCY34Lwrot+EolOMxURjNobRgsKrP9xQOCpTPhW4HWSbwETrbwjuqmmqktbkHVHL0oE0RRWHL3Zqe6G7DRupaZ5HMgeLGwGaCeFnOW0UKCAjCArg3UU3GstIBS2hGwD2Aez8e97ihEhkaeeQetOB2s70OqgPdVF8UGjtNIp+V0g0KIq8b93PmCW+dkZOhfVw2X1kOmHTvizfw4m0gqjzKunKLtK1fN5O59/dkHUWwU90E/utbKWoJrQzc7/RwPdUvH2G10/MJKvCuQv6AKfUYh+uvdsqTYpE9TJhbnC61kcDnqWgTOzDAzUVs0a8lW1CdjOQDN7TaiBPnJVqKGrjQ/96jo/yyx1rJ0X6d2qLQvnqG68GzKZJ1C46642yYoVgIVU6DZ2lAYdgX2r8SQkcKjqQE2uJjC4Y/w6gFwNPmPy8ympkSnaoD6cO+b8PZPVqRoA8UNBSjhBWJRQCdslhHmuiw7M08ApZCZh1xO75nl3QKGINcsU2pYllvzsot4uo5AAmRLGag8/CvI8uWlVMPwaDFXvqYGJmkFUJm5VTrKPKk9DsDAXBx1CFRL5Mue4V65AZJuQ3qtDaSuAAfujx7Z7jlLBnINZcmxE67G0td0KjJDAurgMPedEaXB+o+XndE1D6XpPUeDRRutgLga7HDzt1uJujzWhkwxMRSqMQWVQZMgWt2TRoBdZ+NTlqkdUxXg0AcOHIOTNptZWX+U8Q8YvQdy1AAj/vfaR86F5B4LOeWeutMmlZ9igOwB3OwbKus8epGf7SwWgyqdgmZ8H4uFLXopFjzEWEVudyPMYI0+1LddabBQiqRLz25MzAkJHgmdVLFu7g6oPRFUDus3c7bVycUMyXlP+BsgWkWe/DNurF0oxCQQnr3Z5AwNJW3qGbfdLZRxYolfLbMX/PCjClTKxnqRsOiEsT1p5zyTZIAzLdQmRBEtq1K3sT6FbEID8rEX+KhBrKNEEQvXTsyGw0BiNH7dopOVaT1Nt1nvqX6uc4YEg2gKT/wJSUuH2jtcAhAGnfG826FS0yyWCKDFMmp31EYJXP7ZAcvCS/ulSF9YwsMpbX/lf4+5Ua3SKDw8sX2RwyYFdnNPyZgOyDYIcKJ31E9qo/rofjWk87kNX20ZJvI6LLUQsKFRDi9Ainxx6eHzQ3NSaLyaHvixzjA08cttVKfR6D0EwNYWfpZBQk0z2rQ/J/7cLtwGyUbgaaqNw6/5e4VKdNcdkopwcsj9PN5hKP2z/RuiaFyIOfch6FkY/2V5QnC+xjvLrFOUcRQ/a9tSqoNn7iL18xfIgvMAa7Ne//ZNK3iQOC6kOSXsUxOIUE8Q3mEyndSmt6G6oqI/1Kd+tCh2QDp9iUGwduKiTxTpc7mK4BNjnAotpDHeA0CeW3S1BcReDYpVkVRjbVFHrZKYKCLc/LZuDrDDDKgRq1XZetsboT8zgfDqIyWZW9jwPe83xEBSJnud0RAq8aCh7u6pO4bR01Of9C+e2DpJ3+7QOqk9ERxQIrIMEg6oFYkrTewDbUro1/Y2DsodsTVYzvYjzHOf6/Pm7E6gWqShxMTCZoILXSlp7aJXImxsPhugxkTFUW2dxlJaX5WKDDLHLaZY4bfH4COMaGtZxzpB9KMGb4l+UhYI7r7LIcz7M/At/bY3fEaghdleVvZ8jzDUXSxDcFxenptsjUNWzQaSq8z6hKijXotQQLIFXAtVAk0PK75vFahFrWKpKjjwJcjywb4QKdPH6TPxjD8B7Y8dvuufkSJlGw1BnM6FiSQfhFYkRgG15fsSZtV48to8b7cscfnGV+XRJ4aHXOgbmltKZuecAVfgRmwvcFxHe17D4FxLWW8UKVnlyFYgcWg+maHthiVk8AoOc42quuse08U4W+YJ6x60PvMXj40M91/+hGyIU4s+ePD9mkyP2lKRBV3ooHOAm09eQ89SlmCFdJapWtQNE7TO2wx5ja41wYkbYcPcBR1W7l5PDDeP3mdRh06RIzxM0gq6Hqz7HhtOmp98aWzvNq0SWuQ2tQaq7Q7A+0HdgcBrWTuzkcIPop0fsLZlcjocYIpUo+IMpFMGa5LBlhJ8gfDPGaz3GJtHjuIrfw9lmDj5C/I/Y0zJOohYi+ONNCl5WQNUo+KWfX6XsKxYWiyj7OYOnIsvZlYgLXuaqGtHoJO6uHnXvCB1M3xP05zisX+GpYV2bTNtGpltsZOrYcLaNTLfbiD3g0BrHmMnUmMnBtGEnh7NuXNw+vo6S/TPXU8JQ1QqGffpRcc3oaKNuTBzcGOq2BkhbtSA9fdtP281kxJ5EkcUEGMo+Gkq1OadD3k3rDEeKcNS1rmrj6IeXZ386PvvumX/81ycvXp8ev62RMwGkcR3RBWreS6jMLchzWD25t51zKSdOQcRlSMHtiOEerN1yPuzDAPUQrArW6h2WUO2VE0ErbWrS+qWPaogrz0zcWIRNE9DWEu0dQmalCDkgOkaYrVXaXV8FAyrD3VLcUOyKcXC3RZ1VNXcd8rxg35G+6NJHT233F7Vb0xdGsGqk5AUOogvLZjyaKrt6dnay9y1adL3XC7WruR6jULTVN+4pOxj390Vwta97ddgxh/I20va7yRZg7z1fsMgEiky0yVaXkhSwRJ42whJCugg76IC1rJVf36M2DQ+66ohoe8FtlNKQy/0KOVAKwbgyHY9NbFHebCINESOMqmmDVpoYtVoaaD16oXMdIDfu0VQD9xNURRfWkfw3Hs7xqPckey54cPGpWn7EXqXJDQwfJmXEayHSHPAAh82TLLwAK1/FyxUkK1jxJXFxM+jj1fnpp58cnEoVNHSIoXDUz/wWK2uAkRS+8thksxESzJ3JC2YqbnsuPVsY9EgGYSW8wQjPKtJitL6A1OuS6v3sgvb0FIVa0xYazPbKaSu7zhKav5YdjIhdVyksKte5RFlA2oTibMO6ql0atbzQxCdcSqKbmFWVWXhT3l/T9pBihz4ewYKk+pBk9EQsyzXI4DX1uHhShh/GQDrynOq7GesDGXW/YVF/SFPxTugjPAYMNEXX0dfXcDGyyuKQrnOoq2zOsHEfEd7Uty71096MmjMIHcVqMsbjLH1FL+rZc7V+K57knqOuX4JMKlb5dkYxSuAxNF8EZVJ4Kmqo5bb+LkdFjXuGNWtmusFERan757evXp4Otg9v/KhioHsxpjOWWkszsFweFnjl0tXIR9V+3b7ar7zbg8B7DwOq6gQG6JYr2GHNysE9fLws13OwqWxhlpm6ht0+ovmmAAYNQmVwssCjVKiH+H2CPk4xGfd8mPApq3bNKB6/4f1AxS/9Q46lfa+9XgEoJ5Ij1YIXPBeOkTv11MJ3HvT1geVu9hdrr+OcY4zf+AGCNeiw3sOmVr2R3UHFY9sKBk2251OISsIVmNlm+ZhvDbByaiAdtQ5W602bmvshM/zYezV9NYZaV92LqpuVFDAuAl8+Xev1fbob7fv0aZXvHJkPt0Dr/wEYwjVWPDgAAA==",
    "configs/gemma4-12b.yaml": "H4sIAMlDR2oC/3VTy27bMBC8+ysWyMVGIleyndRVkUPzRNGkQV9ngpZWEhuKq5CUHfdQ9CP6hf2SLhk7VVH0JM5wd3Z3uDqAC6xkr30OD73Uym+hpRJhbE/nsyOQumsknyZTuJLOP98t91fLCSg3OoCCjLekNZaw2oJvEAx5XBHdw+3dxeWNOL97f/X2+hOUqvCvgdZorWKttZKQJFG24gJTlrrGtpWwgGx2Bl+MqhRrnpOpVN1b6RUZ+PXjJ984Tb4BiwW1LZoSy/ypJzg9BQvjLM8mrHZpCha3SWURoeVBFReTGsYeHz0cgmpljfyVfalocgSz45N3cRq+5vRFslIePtzQxzfAlT9zWyfXZ5yt6oY9+56lSTZjYqO4lxh1CNRxEfUNbah/y9V1DjVRrfFFHWZLFpxzlrDu2BB0FhO23njOKGEtreIzVJba/YyTURtFRgBGtvgfMb710tboBUf3Gl2IB0jgQXSWvu7A/RCsh4CGoJYeh7jvhqikjdljm8N8xl9NVoro/5AoLXXUs1EpMyslXc57YZBBKx+Fwweh0dS+yWGRvjoJMdIXjXDsRQ5ZVJGlUEYs+Bly8LYPub1DYV0o8BdVW1kqNF4UDRb3HSn21NQ59E82jkbeSmUCFZzsWxGxwI6Khhubh3IobYgQvGncwWyaYrJgfiNtyx7E/eNhpulxIDEsgSixkNtIptlusNCJMGTbQEddKxx3Fd7FCr/tWLsgp6ITBivfG+R4xUPsLAz6z/PIouh5dePyC+ex426XHBAXjbe+lO1GLFdxBxyGH2G+SF8GINfICWGYmluMqX/oqJOl6Z7x5KUWmneXrQ5PiGvG/6bv6EF6fKUVurh7qIX0gv/I58dp0VtViIrsICh/0tHk3Og3BV0gCIYEAAA=",
    "configs/gemma4.yaml": "H4sIAMlDR2oC/2VTy27bMBC8+ysWyCUGaleO1TRVkEPeKNA0aPq4EmtpLbGmSIVc2XVP/Yh+Yb+kS9pxBfRkzXB39jHrI7ihJfaGC3ju0WjeQusqgmN/MT95BWi6BuVrPIU7DHx4O3t5OhuDDqMjKJ1l74yhChZb4IbAOqaFcyt4eLy5/aCuHz/evb//DJUu+RzcmrzXorXWCJNJkl1KgalI3VPbIuRwm1/BtbNLXfceWTsLf379hq82GMcNeCpd25KtqCp2vcDFBXg4nhWzsajE7I3zqwCSuPRE8CWXx9P7K/j2dPkwHsWaphgBWGypgH6n+7qO1Sf5RPInmuWZ0dfESsJ7QyEmAEzgWXXefd+D1RCsh8ANQY1MQ9x3Q1S5jX3BvoD5ifwa51Gl4YZE5V3nenEsE2ahMRSybEsCWvyhAj0rQ7bmpoA8e3caY5DLRgX9U+acJRWslLYqX2hRYd8TwBHkE4Hw6YN7uoTAaCv0laRU5xCIxR0TxCPnYXaaAmOcaPWBlA+xr53Snqo9Vposq7KhctU5bVnb+rDm0Yg9ahup6EDfqoQVda5sZJ557JLQxwgl7kvjJ9OMJrnwG/StrC7dhOxgmr2JJOm6YVVRidtEZrP9PmInyjrfRjrpehWkq2inV7ztRLt0QacFWlpyb0nitQyx33zUP8yDZdm3vUkHqQJTJ93GrlzHWkpghe1GnS3S6QSKxznPs7cR4JokIQ5TS4sp9R+ddGZZ9sKwYzTK6DY6FJ2nteD/0/f0ID2Zu6CQTpaMQlbyLzmY0xJ7XSoxchBU7HSMC2H0F9K1I5MSBAAA",
    "configs/ornith10.yaml": "H4sIAMlDR2oC/2VSy27bMBC8+ysWyDVy5UfSRremRYGiCYoE6ZlYU2uJDUUy5MqOe+pH9Av7JV3STqqiJ2FGO8uZ3T2Dr9EZ7mExr+HqGj54tzXdGJGNd/D75y8IPnHFEY2jFrbRD3C3Jwer+QVsDvCRKNyTcVsfNc3O4PbzA1ijySU6h+Xl8gto75ie+bzIqg0maaN7ZGAagkXOqm8uWc99pf0Q5OWNJdgZhE+Y+AZdN2JHt74lC/tsdV1tDMPdjb9/D2LyYT0b8s9mBuBwoAZaMRVfTFVo3hwzVpKxurqWMsbYESuRjZZSFgJU8KRC9N9P4HEKdlPgp6CTAFM8hilq/d694NjAailf6yMqtKHHKdFGH/zIDdTCbAymBpx3JGDAZ5XoSVlyHfcNrOury1yDrHuVzA/JuyhdsFXGqbXMpgGOY9aOiVRM+YF/qC5ia8ix0j3px+CNY+O6BsbjGmazsu5M5ZGOgypYUfC6F2Or/BxhzBVKLkUcLOc1VWvh9xgHmUG5Hwkzry8ySabrWbWk8VDIenEKlp0o5+OQ6dI3qiSu8l6i4kOQ3tonUybhaMujI6k3EuI0wtz/NQ9qPQ6jLcerElMQt9mVD2zkCWxx2Kt3MiHhElErWdb12wxwRyLIYTqxWKR/6dJnUdcvDHtGq6wZ8qjzCmkn+H/5iZ7Iy5Y2lMrtkVXIilz7upyBOBqt5G4nRc2xj/Upzf4AbbYxc68DAAA=",
    "configs/qwen35-4b.yaml": "H4sIAMlDR2oC/2VSwU7cQAy971dY4gISu01goZCKC1CqSm1RRXseeRNvMt3JTPA4u11O/Yh+Yb+knrAsqXpK3rP9/GzPAdzSEnsnBTz26KxsoQ0VwSFf5efHgK5rUP+OZnCHUfaxi5fQxRHYODmAMnjh4BxVsNiCNAQ+CC1CWMHn+9v3n8zN/Ze7jx8eoLKlvIOwJmarWmuLMJ0OskttMFOprxvycDo7g/k13AS/tHXPKDZ4+PPrN3z30QVpgKkMbUu+oqp49gJXV8BwmBf5kao8tKhuWENC7LV+TSDhVfzy+hg2gVcRaiYUUPlv80ky4ooJgMeWCuifm71JVVo0nV9rSJBrEqOpvaOYkgGm8Gg6Dj92YDUG6zEIY1Cj0Bj33RhVYeNfMBeQn+vXBUYzTDsmKg5d6PWEmTILi7HQ7XtS0OJPE+nROPK1NAXMs8tUtUApGxPtk86YDypYGevNfGFVRbhPtX0kwzE1+IeqGStLXkzZULnqgvVifb3f1WQijNYnKq2xb82ADXWhbNTYaWpHyCnD6F3Vwckso+lc+Q1yqzsYrq3DzLKzRJKtGzEVlbgdyCzfDZacGB+4TfSgyyaqq3QXNrLtVLsM0Q6b8LSU3pPmWx1it8Kkv58Hy7Jvezc8NROFOnWbXIVOrLbACtuNudANKRcpPbvTefY2AVyTFqRharU4lL7Sg06eZS+MBEFnnG3Tqk+UpbXi/8t39Kh8uNKC4vD2yBkUo+9/f5yWhG1ploFHScWzjgsxTv4CrIUJ2+wDAAA=",
    "configs/qwen35.yaml": "H4sIAMlDR2oC/2VTwW4TMRC95ytG6qWRSNg0aUkX5UBbWiEVKqCcLWd3smvi9Wzt2aThgPgIvpAvYeym6SJO2fc8fvPmjXMEV7jSneUcHjptDe+goRLh2C+mJ69A27bW8jUcw7UOfDibPx/Nh2DC4AgKcuzJWixhuQOuERwxLonW8PHu6v2turz7dP3h5iuUpuC3QBv03ojWxmgYjZLsShqMRerzFh1Mx6dwfgGX5Fam6rxmQw7+/PoN31ywxDV4LKhp0JVY5k9eYLEAD8eTfDKMKrf05R1sjZRa0qUyTs2Whhf3vkPYkl8HEMX7GRz/PLu5gC2aquYwHEQrNh8AON1gDt1Tu9fRlHganV/IEWtfISsp7SyGWAwwggfVevq+B+s+2PQB9UGlGfu4a/uopK17xj6H6Yn8WvJapXn7ROmppU6WmAmzNDrkkr9DAY1+VAEflEVXcZ3DLDs/izWai1oF80NmnCSVl4xyYAlJyC6g8iE2+IeqvC4NOlZFjcW6JePYuOqQ1WDAXhsXqRhj16iEFbZU1GJsGtuh9rFCyWbFwck4w9FM+K32jWSQ9i3DjLPTSKbdqBILvUtkNtkPFp0oR76JdNL1KoiruBeveNeKdkHBpCQcrrhzKPVGhthHGPUP8+ii6JrOpsemAmMrbudSQC0baaFL3WzVXBISLmB8eNNZ9iYCvUG5EIepxGK6+kInnUmWPTNMrK2ypolRxxXiRvD/1/d073ra0hJDentolWYl/4DDchpkbwq1It8ryp90LIUw+AsCiSQL7gMAAA==",
    "config.yaml": "H4sIAMlDR2oC/+1YTY/bNhC961cMsJemgGR5V9tudGt2kyBos4u2KXokRtJYYkyRWpKy4/z6DmVJ9bpdtIcUCFDdqDfDx/l6ks0LeCM1+V5LXcOt0RtZ9xa9NDq6gN8c1pRDstocfShxDdTUtphBHKNSETvd0QZ75aE1FamoOj6J4SkfnYPb7z/8cv/u/m0O5XBIcsBWwQ5VTw7QEjz2qKQ/xGEjjCwugQ8NgTaeCmO2zPL+4e71T+L24f7Nu7e/QiVLD501O1kxS0f2uNvsyNoB+maDzq9G6hcJE/xI1EFhfANSgzvoEvYNacCqCgVA0LQ/ZhKc34fFGPBYFeZspbXGjrBbfXtMxXnbl77nTJyBDdemwHILe2O37kU0MLo8grEgYQXHc45LAI0tl7rXTnFwq8EtzuLX2atY+tHFo61pKG2vyE0bAWJ4FFyFjyfA9hzYnQPmHKjR0znWd+dIZfb6FLM5XF2Oa2UsClRdg+dgZU1nep9DOqKFRJdzYzWNQIufhKNHoUjXvskhS19+N/miLxvh5Geuz3pmxUpILbJCMiuXfuLpHQnrwqF/gWuLlSTtRdlQue2M1J57Ptd88PQWZZDC3JW+FQMmqDNlwzFfTREQ2uApeC44sMskpTgbbXu0LZdumBjOOUmvJwPJuvGiohIPgyFdn+QfAhTa2DaY5nOscBxwaLkV/tBRUJCTc+E0bYIyeZ/kHMfqT+fNKWNZ9m2vhhEWzlPHmUzRms5LPhIrbPfippjHzRFVnG+Wfj8BuCPeHBKuOfyB5qlp4F2n6SnqjUcllGxDp6apIFb+M1Sj6YxqaHhBbnyzCPSCdPWkxy15K0uxMfbEMT/yKeNcNMsvXl8Wz0qwNqZWNCtwfbkocFHgf6PAm/+hAh/3pK+u//ED+HNwS67jl68W6S3SW6T35aQXZ8W/Vl/2lavv5m/Ed7Nob/nh+ZVpjx0NN8036/RZ7VX8z9SS1ExTUoxy9TDsiNdJunwFFyUuX8EvosQLuEOPT+90ooqh0G6Le1FJnucArPiJMR70kpyj6sQyY1E4khV5ajwCf1o2UtET02qYpuSjM3q4QLs1CouziMqADfdFXZ/DhzBVjk8MreOXxEZ+ymG6k4v+AIAKVefDEwAA",
}

for rel_path, encoded in encoded_files.items():
    dest = f"/content/{rel_path}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    raw = base64.b64decode(encoded)
    data = gzip.decompress(raw)
    with open(dest, "wb") as f:
        f.write(data)
    print(f"  Wrote {rel_path}")

sys.path.insert(0, "/content")
print(f"\nDecoded {len(encoded_files)} files")


## Section 3: Choose Model + Mode


In [ ]:
# ============================================================
# 3.0 Select model and mode
# ============================================================
# MODEL = "gemma4"      -> Gemma 4 E4B (unsloth)
# MODEL = "gemma4-12b"  -> Gemma 4 12B (google)
# MODEL = "qwen35"      -> Qwen 3.5 9B (unsloth)
# MODEL = "qwen35-4b"   -> Qwen 3.5 4B (unsloth)
# MODEL = "ornith10"    -> Ornith 1.0 9B (deepreinforce)
# MODE = "fast"     -> 1 epoch, ~30 min
# MODE = "quality"  -> 3 epochs, ~50-70 min
MODEL = "qwen35-4b"
MODE = "fast"

# ============================================================
# 3.1 Load config + apply mode overrides
# ============================================================
import yaml
with open(f"/content/configs/{MODEL}.yaml") as f:
    cfg = yaml.safe_load(f)
mc = cfg.get("model", cfg)
tc = cfg.get("training", {})

MODEL_NAME = mc["name"]
LORA_R = mc["r"]
LORA_ALPHA = mc["lora_alpha"]
MAX_SEQ_LENGTH = mc.get("max_seq_length", 4096)
NUM_EPOCHS = tc.get("num_train_epochs", 3)
GRAD_ACCUM = tc.get("gradient_accumulation_steps", 4)
WARMUP_RATIO = tc.get("warmup_ratio", 0.05)
USE_RSLORA = mc.get("use_rslora", True)
NEFTUNE_ALPHA = tc.get("neftune_noise_alpha", 5)

LEARNING_RATE = float(tc.get("learning_rate", 2e-4))
if MODE == "fast":
    LORA_R = min(LORA_R, 8)
    LORA_ALPHA = min(LORA_ALPHA, 16)
    MAX_SEQ_LENGTH = min(MAX_SEQ_LENGTH, 2048)
    NUM_EPOCHS = 1
    GRAD_ACCUM = min(GRAD_ACCUM, 4)

CHAT_TEMPLATE = "gemma-4" if MODEL.startswith("gemma") else "chatml"
OUTPUT_DIR = f"{WORK_DIR}/outputs/{MODEL}-ctf"

print(f"\n{MODEL.upper()} / {MODE.upper()}")
print(f"  LoRA: r={LORA_R}, a={LORA_ALPHA}, rsLoRA={USE_RSLORA}")
print(f"  Epochs: {NUM_EPOCHS} | Grad accum: {GRAD_ACCUM} | Seq: {MAX_SEQ_LENGTH}")
print(f"  Chat: {CHAT_TEMPLATE}")
est = "~30 min (FAST)" if MODE == "fast" else "~50-70 min (QUALITY)"
print(f"  Est: {est}")


## Section 4: Build Dataset


In [ ]:
# ============================================================
# 4.0 Build dataset — import from src/
# ============================================================
from src.build_dataset import build_writeups_dataset, build_docs_dataset, build_ctfdojo_dataset
from src.download_datasets import (download_ctf_webserver, download_opencode_reasoning,
    download_fenrir, download_ctf_solver, download_summermc_ctf,
    download_trendyol_cybersec, download_ctf_crypto_analysis)
from src.synthetic_rev_pwn import save_to_file
from src.process_data import convert_alpaca_to_chat
import glob

for d in [f"{WORK_DIR}/data/raw", f"{WORK_DIR}/data/processed", f"{WORK_DIR}/data/merged"]:
    os.makedirs(d, exist_ok=True)

print("Building datasets...")
t0 = time.time()
mr = 30 if MODE == "fast" else 999999
ms = 200 if MODE == "fast" else 5000
build_writeups_dataset(f"{WORK_DIR}/data/raw/writeups.jsonl", mr)
build_docs_dataset(f"{WORK_DIR}/data/raw/docs.jsonl", 5 if MODE == "fast" else 999999)
build_ctfdojo_dataset(f"{WORK_DIR}/data/raw/ctfdojo.jsonl", 200 if MODE == "fast" else 500)
save_to_file(f"{WORK_DIR}/data/raw/synthetic_rev_pwn.jsonl")
download_ctf_webserver(f"{WORK_DIR}/data/raw/ctf_webserver.jsonl")
download_opencode_reasoning(f"{WORK_DIR}/data/raw/opencode_reasoning.jsonl", ms)
download_fenrir(f"{WORK_DIR}/data/raw/fenrir_cybersecurity.jsonl", ms)
download_ctf_solver(f"{WORK_DIR}/data/raw/ctf_solver.jsonl", ms)
download_summermc_ctf(f"{WORK_DIR}/data/raw/summermc_ctf.jsonl", ms)
download_trendyol_cybersec(f"{WORK_DIR}/data/raw/trendyol_cybersec.jsonl", ms)
download_ctf_crypto_analysis(f"{WORK_DIR}/data/raw/ctf_crypto_analysis.jsonl", ms)
print(f"  Build: {time.time()-t0:.1f}s")

print("Processing to ChatML...")
t0 = time.time()
for rf in glob.glob(f"{WORK_DIR}/data/raw/*.jsonl"):
    name = Path(rf).stem; n = 0
    with open(rf) as fi, open(f"{WORK_DIR}/data/processed/{name}.jsonl", "w") as fo:
        for line in fi:
            item = json.loads(line); out = item.get("output", "")
            if not out: continue
            r = convert_alpaca_to_chat(item.get("instruction",""), item.get("input",""),
                out, item.get("category","ctf"))
            fo.write(json.dumps(r) + "\n"); n += 1
        print(f"  {name}: {n}")

merged = f"{WORK_DIR}/data/merged/train.jsonl"
total = 0
with open(merged, "w") as fo:
    for pf in glob.glob(f"{WORK_DIR}/data/processed/*.jsonl"):
        with open(pf) as fi:
            for line in fi:
                fo.write(line); total += 1
print(f"Merged {total} examples")
print(f"  Process: {time.time()-t0:.1f}s")


## Section 5: Load Model + Train


In [ ]:
# ============================================================
# 5.0 Load model with Unsloth
# ============================================================
from unsloth import FastLanguageModel, get_chat_template

print(f"Loading {MODEL_NAME}...")
t0 = time.time()

if MODEL.startswith("gemma"):
    from unsloth import FastVisionModel
    model, processor = FastVisionModel.from_pretrained(
        model_name=MODEL_NAME, load_in_4bit=True,
        use_gradient_checkpointing="unsloth")
    tokenizer = processor.tokenizer
    tokenizer = get_chat_template(tokenizer, "gemma-4")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True, dtype=None)
    tokenizer = get_chat_template(tokenizer, "chatml")
print(f"  Loaded in {time.time()-t0:.1f}s")

# ============================================================
# 5.1 Configure LoRA
# ============================================================
print("LoRA...")
t0 = time.time()
TGTS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

if MODEL.startswith("gemma"):
    from unsloth import FastVisionModel
    model = FastVisionModel.get_peft_model(model, finetune_vision_layers=False,
        finetune_language_layers=True, r=LORA_R, lora_alpha=LORA_ALPHA,
        lora_dropout=0, bias="none", use_rslora=USE_RSLORA)
else:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(model, r=LORA_R,
        target_modules=TGTS, lora_alpha=LORA_ALPHA, lora_dropout=0,
        bias="none", use_gradient_checkpointing="unsloth",
        use_rslora=USE_RSLORA)
print(f"  LoRA r={LORA_R}, a={LORA_ALPHA}, rsLoRA={USE_RSLORA} ({time.time()-t0:.1f}s)")

# ============================================================
# 5.2 Load dataset
# ============================================================
print("Loading dataset...")
t0 = time.time()
ds = load_dataset("json", data_files={"train": merged}, split="train")
ds = ds.filter(lambda ex: bool(ex.get("messages",[])) and bool(ex["messages"][-1].get("content","")))
print(f"  {len(ds)} examples")

at = tokenizer.tokenizer if hasattr(tokenizer,"tokenizer") else tokenizer
def len_ok(ex):
    t = at.apply_chat_template(ex["messages"], tokenize=False)
    return len(at.encode(t)) <= MAX_SEQ_LENGTH
before = len(ds); ds = ds.filter(len_ok, desc="Filter")
print(f"  {len(ds)} (dropped {before-len(ds)} long)")

split = ds.train_test_split(test_size=0.1, seed=42)
td, ed = split["train"], split["test"]
print(f"  Train: {len(td)}, Eval: {len(ed)} ({time.time()-t0:.1f}s)")


In [ ]:
# ============================================================
# 5.3 Train with SFTTrainer
# ============================================================
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

class ProgressCB(TrainerCallback):
    def __init__(self):
        self.pbar = None; self.start = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.time()
        self.pbar = tqdm(total=state.max_steps, desc="Training", unit="step")
        mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
        print(f"VRAM: {mem:.1f}GB/{torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.pbar and logs:
            self.pbar.set_postfix(loss=f"{logs.get('loss',0):.4f}")
            self.pbar.update(1)
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar: self.pbar.close()
        if self.start: print(f"Done in {(time.time()-self.start)/60:.1f} min")

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=td, eval_dataset=ed,
    args=SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH, per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=WARMUP_RATIO,
        num_train_epochs=NUM_EPOCHS, learning_rate=float(LEARNING_RATE),
        weight_decay=0.001, lr_scheduler_type="cosine", logging_steps=10,
        output_dir=OUTPUT_DIR, optim="adamw_8bit", seed=3407,
        save_strategy="steps", save_steps=100, save_total_limit=2,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        assistant_only_loss=True, report_to="none",
        eval_strategy="steps", eval_steps=100,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
    ),
    callbacks=[ProgressCB()],
)

print(f"\nTraining ({NUM_EPOCHS} epochs, bs={GRAD_ACCUM})")
trainer.train()

print("\n=== Saving LoRA ===")
lp = f"{OUTPUT_DIR}/lora"
model.save_pretrained(lp); tokenizer.save_pretrained(lp)
print(f"  Saved {lp}")


## Section 6: Export + Evaluate


In [ ]:
# ============================================================
# 6.0 Export models
# ============================================================
if not MODEL.startswith("gemma"):
    print("GGUF...")
    model.save_pretrained_gguf(f"{OUTPUT_DIR}/gguf", tokenizer, quantization_method="q4_k_m")
    print(f"  {OUTPUT_DIR}/gguf")
    print("Merged...")
    model.save_pretrained_merged(f"{OUTPUT_DIR}/merged", tokenizer, save_method="merged_16bit")
    print(f"  {OUTPUT_DIR}/merged")
else:
    print("Skip GGUF/merged (FastVisionModel)")


In [ ]:
# ============================================================
# 6.1 Evaluate
# ============================================================
print("\n=== Evaluation ===\n")
import subprocess
r = subprocess.run(["python", "src/eval.py", "--model", MODEL, "--adapter", lp],
    capture_output=True, text=True, cwd=WORK_DIR)
print(r.stdout[-2000:] if len(r.stdout) > 2000 else r.stdout)
if r.returncode: print(f"Eval error: {r.stderr[-300:]}")
